In [2]:
import numpy as np
import pandas as pd
from pyhive import presto
from datetime import timedelta,datetime
import json
import requests
import math
import os


pd.set_option('display.max_columns', None)
import warnings
warnings.filterwarnings("ignore")
conn = presto.connect(
        host="presto-gateway.serving.data.plectrum.dev",
        port=80,
        username="manoj.ravirajan@rapido.bike"
    )
presto_port = '80'
username = 'manoj.ravirajan@rapido.bike'
conn1 = presto.connect('bi-presto.serving.data.production.internal', presto_port, username)
conn2 = presto.connect('bi-trino-2.serving.data.production.internal', presto_port, username)
conn3= presto.connect('processing-2.processing.data.production.internal',presto_port,username)
conn4= presto.connect('processing.processing.data.production.internal',presto_port,username)

### New Segment

In [31]:
cx_data0 = f'''
    with base as (
    select 
        customer_id, 
        segment  
    from 
        hive.experiments_internal.customer_segment_amj25
    ),
    
    orders as (
    
    select
        date_format(date_trunc('week', date_parse(yyyymmdd, '%Y%m%d')), '%Y%m%d') week_start_date,
        customer_id,
        count(distinct order_id) gross_orders,
        count(distinct case when order_status = 'dropped' then order_id end) net_orders
    from 
        orders.order_logs_fact
    where
        yyyymmdd >= '20241216'
        and yyyymmdd <= '20250316'
        and city_name = 'Bangalore'
        and service_obj_service_name in ('Auto','Auto Lite','Auto Pool','AutoPremium','CityAuto','Link','Bike Lite','Bike Pink','CabEconomy','Bike Metro', 'CabPremium')
    group by 1,2
    )
    
    
    select
        segment,
        sum(case when week_start_date = '20241216' then gross_orders end ) w01_gross,
        sum(case when week_start_date = '20241223' then gross_orders end ) w02_gross,
        sum(case when week_start_date = '20241230' then gross_orders end ) w03_gross,
        sum(case when week_start_date = '20250106' then gross_orders end ) w04_gross,
        sum(case when week_start_date = '20250113' then gross_orders end ) w05_gross,
        sum(case when week_start_date = '20250120' then gross_orders end ) w06_gross,
        sum(case when week_start_date = '20250127' then gross_orders end ) w07_gross,
        sum(case when week_start_date = '20250203' then gross_orders end ) w08_gross,
        sum(case when week_start_date = '20250210' then gross_orders end ) w09_gross,
        sum(case when week_start_date = '20250217' then gross_orders end ) w10_gross,
        sum(case when week_start_date = '20250224' then gross_orders end ) w11_gross,
        sum(case when week_start_date = '20250303' then gross_orders end ) w12_gross,
        sum(case when week_start_date = '20250310' then gross_orders end ) w13_gross,
        
        sum(orders.gross_orders) as overall_gross,
        100.00*sum(gross_orders)/sum(sum(gross_orders))over() as pct,
        
        sum(case when week_start_date = '20241216' then net_orders end ) w01_net,
        sum(case when week_start_date = '20241223' then net_orders end ) w02_net,
        sum(case when week_start_date = '20241230' then net_orders end ) w03_net,
        sum(case when week_start_date = '20250106' then net_orders end ) w04_net,
        sum(case when week_start_date = '20250113' then net_orders end ) w05_net,
        sum(case when week_start_date = '20250120' then net_orders end ) w06_net,
        sum(case when week_start_date = '20250127' then net_orders end ) w07_net,
        sum(case when week_start_date = '20250203' then net_orders end ) w08_net,
        sum(case when week_start_date = '20250210' then net_orders end ) w09_net,
        sum(case when week_start_date = '20250217' then net_orders end ) w10_net,
        sum(case when week_start_date = '20250224' then net_orders end ) w11_net,
        sum(case when week_start_date = '20250303' then net_orders end ) w12_net,
        sum(case when week_start_date = '20250310' then net_orders end ) w13_net,
        
        sum(orders.net_orders) as overall_gross,
        100.00*sum(net_orders)/sum(sum(net_orders))over() as pct

    from
        base
    left join 
        orders
        on base.customer_id = orders.customer_id
    group by 1
    order by 1
       
'''
cx_data0 = pd.read_sql(cx_data0, conn4)
cx_data0.head()

,segment,w01_gross,w02_gross,w03_gross,w04_gross,w05_gross,w06_gross,w07_gross,w08_gross,w09_gross,w10_gross,w11_gross,w12_gross,w13_gross,overall_gross,pct,w01_net,w02_net,w03_net,w04_net,w05_net,w06_net,w07_net,w08_net,w09_net,w10_net,w11_net,w12_net,w13_net,overall_gross,pct
0,1-DAILY,978303,732304,828267,1104298,1015273,1197754,1172648,1252083,1281503,1136586,1117268,1189747,1086237,14092271,22.56,643553,532162,591490,702954,666446,728138,749421,745258,736039,729555,707487,714627,682575,8929705,22.74
1,2-WEEKLY,1029757,801451,901391,1110973,1032780,1185439,1157025,1253815,1296343,1136923,1131776,1201827,1133625,14373125,23.01,657600,565326,625494,699825,667488,707954,719626,732853,733061,711421,700828,709856,697665,8928997,22.74
2,3-MONTHLY,1912620,1540358,1708287,1950871,1935235,2152872,2150166,2496496,2492256,2093442,2095683,2220009,2203215,26951510,43.15,1213231,1070538,1149882,1244925,1243132,1301369,1350608,1487036,1432724,1328451,1316692,1333680,1379189,16851457,42.92
3,4-OTHER,421694,342426,332751,280318,254822,304261,301702,313374,477777,549158,619018,745326,810159,5752786,9.21,268015,234211,219685,180279,168880,191867,196781,195535,298530,367141,407326,468705,529159,3726114,9.49
4,5-INACTIVE,93133,77267,86830,91177,88082,98514,99608,111225,114783,103540,105653,111969,112330,1294111,2.07,60001,54443,58960,58965,57077,60997,63744,67498,67772,67238,67346,69411,72061,825513,2.10


In [13]:
cx_data0

,segment,w01_gross,w02_gross,w03_gross,w04_gross,w05_gross,w06_gross,w07_gross,w08_gross,w09_gross,w10_gross,w11_gross,w12_gross,w13_gross,overall_gross,pct,w01_net,w02_net,w03_net,w04_net,w05_net,w06_net,w07_net,w08_net,w09_net,w10_net,w11_net,w12_net,w13_net,overall_gross,pct
0,1-DAILY,525448,410183,462315,583872,539047,620750,609391,645308,660706,590417,584798,614802,566184,7413221,11.87,352220,303092,334537,379730,360208,386012,396721,392675,388656,387573,377496,378049,364055,4801024,12.23
1,2-WEEKLY,1468247,1111724,1254550,1614265,1487610,1730882,1687116,1824895,1882465,1654254,1635418,1746836,1626346,20724608,33.18,939622,786050,873780,1012242,960050,1031207,1051170,1064398,1060765,1034406,1012300,1028246,998967,12853203,32.74
2,3-MONTHLY,1926985,1552206,1721080,1968005,1956631,2184433,2183332,2532191,2526931,2122280,2124511,2249945,2230547,27279077,43.67,1222542,1078884,1158549,1255732,1256808,1320242,1371764,1508074,1452403,1347448,1335211,1351868,1396407,17055932,43.44
3,4-OTHER,421694,342426,332751,280318,254822,304261,301702,313374,477777,549158,619018,745326,810159,5752786,9.21,268015,234211,219685,180279,168880,191867,196781,195535,298530,367141,407326,468705,529159,3726114,9.49
4,5-INACTIVE,93133,77267,86830,91177,88082,98514,99608,111225,114783,103540,105653,111969,112330,1294111,2.07,60001,54443,58960,58965,57077,60997,63744,67498,67772,67238,67346,69411,72061,825513,2.10


In [33]:
cx_data0.to_clipboard(index=False)

### Office bins

In [3]:
cx_data15 = f'''
    with base as (
    
        select
            customer_id,
            segment,
            -- total_net_orders,
            -- office_orders,
        /*
        case 
        when office_orders = '0' then 'a. 0 office ride'
        when office_orders = '1' then 'b. 1 office ride'
        when office_orders >= '2' and office_orders <= '3' then 'c. 2-3 office'
        when office_orders >= '4' and office_orders <= '6' then 'd. 4-6 office'
        when office_orders >= '7' and office_orders <= '10' then 'e. 7-10 office'
        when office_orders > '10' then 'f. 10+ office'
        else 'a. 0 office ride'
        end office_bin
    
        case 
        when office_orders = '0' and transit_orders = '0' then 'a. no-office-transit'
        when office_orders = '1' then 'b. 1 office ride'
        when office_orders >= '2' and office_orders <= '3' then 'c. 2-3 office'
        when office_orders >= '4' and office_orders <= '6' then 'd. 4-6 office'
        when office_orders >= '7' and office_orders <= '10' then 'e. 7-10 office'
        when office_orders > '10' then 'f. 10+ office'
        when transit_orders = '1' then 'g. 1 transit ride'
        when transit_orders >= '2' and transit_orders <= '3' then 'h. 2-3 transit ride'
        when transit_orders > '3' then 'i. 3+ transit ride'
        else 'a. no-office-transit'
        end office_bin
        */
        
        case 
        when office_orders = '0' and transit_orders = '0' then 'a. no-office-transit'
        when office_orders >= '1' then 'b. office ride'
        when transit_orders >= '1' then 'c. transit ride'
        else 'a. no-office-transit'
        end office_bin
        
        
    from 
        hive.experiments_internal.customer_base_office_transit_tag_amj2025
    ),
    
    orders as (
    
    select
        date_format(date_trunc('week', date_parse(yyyymmdd, '%Y%m%d')), '%Y%m%d') week_start_date,
        customer_id,
        count(distinct order_id) gross_orders,
        count(distinct case when order_status = 'dropped' then order_id end) net_orders
    from 
        orders.order_logs_fact
    where
        yyyymmdd >= '20241216'
        and yyyymmdd <= '20250316'
        and city_name = 'Bangalore'
        and service_obj_service_name in ('Auto','Auto Lite','Auto Pool','AutoPremium','CityAuto','Link','Bike Lite','Bike Pink','CabEconomy','Bike Metro', 'CabPremium')
    group by 1,2
    )
    
    
    select
        office_bin,
        sum(case when week_start_date = '20241216' then gross_orders end ) w01_gross,
        sum(case when week_start_date = '20241223' then gross_orders end ) w02_gross,
        sum(case when week_start_date = '20241230' then gross_orders end ) w03_gross,
        sum(case when week_start_date = '20250106' then gross_orders end ) w04_gross,
        sum(case when week_start_date = '20250113' then gross_orders end ) w05_gross,
        sum(case when week_start_date = '20250120' then gross_orders end ) w06_gross,
        sum(case when week_start_date = '20250127' then gross_orders end ) w07_gross,
        sum(case when week_start_date = '20250203' then gross_orders end ) w08_gross,
        sum(case when week_start_date = '20250210' then gross_orders end ) w09_gross,
        sum(case when week_start_date = '20250217' then gross_orders end ) w10_gross,
        sum(case when week_start_date = '20250224' then gross_orders end ) w11_gross,
        sum(case when week_start_date = '20250303' then gross_orders end ) w12_gross,
        sum(case when week_start_date = '20250310' then gross_orders end ) w13_gross,
        
        sum(orders.gross_orders) as overall_gross,
        100.00*sum(gross_orders)/sum(sum(gross_orders))over() as pct,
        
        sum(case when week_start_date = '20241216' then net_orders end ) w01_net,
        sum(case when week_start_date = '20241223' then net_orders end ) w02_net,
        sum(case when week_start_date = '20241230' then net_orders end ) w03_net,
        sum(case when week_start_date = '20250106' then net_orders end ) w04_net,
        sum(case when week_start_date = '20250113' then net_orders end ) w05_net,
        sum(case when week_start_date = '20250120' then net_orders end ) w06_net,
        sum(case when week_start_date = '20250127' then net_orders end ) w07_net,
        sum(case when week_start_date = '20250203' then net_orders end ) w08_net,
        sum(case when week_start_date = '20250210' then net_orders end ) w09_net,
        sum(case when week_start_date = '20250217' then net_orders end ) w10_net,
        sum(case when week_start_date = '20250224' then net_orders end ) w11_net,
        sum(case when week_start_date = '20250303' then net_orders end ) w12_net,
        sum(case when week_start_date = '20250310' then net_orders end ) w13_net,
        
        sum(orders.net_orders) as overall_gross,
        100.00*sum(net_orders)/sum(sum(net_orders))over() as pct

    from
        base
    left join 
        orders
        on base.customer_id = orders.customer_id
    group by 1
    order by 1
       
'''
cx_data15 = pd.read_sql(cx_data15, conn3)
cx_data15.head()

,office_bin,w01_gross,w02_gross,w03_gross,w04_gross,w05_gross,w06_gross,w07_gross,w08_gross,w09_gross,w10_gross,w11_gross,w12_gross,w13_gross,overall_gross,pct,w01_net,w02_net,w03_net,w04_net,w05_net,w06_net,w07_net,w08_net,w09_net,w10_net,w11_net,w12_net,w13_net,overall_gross,pct
0,a. no-office-transit,1095585,889043,962759,1030102,1022487,1114597,1105287,1258681,1317859,1170104,1197008,1289719,1350695,14803926,23.70,666104,591189,622491,637500,633557,657941,676143,731607,747780,731574,738853,772263,838067,9045069,23.04
1,b. office ride,2704099,2063373,2296376,2869861,2664161,3140224,3094706,3411848,3562212,3148195,3160640,3430583,3252421,38798699,62.11,1758228,1478582,1610594,1827139,1749136,1897975,1959193,2025764,2044328,2007511,1996628,2051352,2038312,24444742,62.26
2,c. transit ride,635823,541390,598391,637674,639544,684019,681156,756464,782591,701350,711750,748576,742450,8861178,14.19,418068,386909,412426,422309,420330,434409,444844,470809,476018,464721,464198,472664,484270,5771975,14.70


In [4]:
cx_data15

,office_bin,w01_gross,w02_gross,w03_gross,w04_gross,w05_gross,w06_gross,w07_gross,w08_gross,w09_gross,w10_gross,w11_gross,w12_gross,w13_gross,overall_gross,pct,w01_net,w02_net,w03_net,w04_net,w05_net,w06_net,w07_net,w08_net,w09_net,w10_net,w11_net,w12_net,w13_net,overall_gross,pct
0,a. no-office-transit,1095585,889043,962759,1030102,1022487,1114597,1105287,1258681,1317859,1170104,1197008,1289719,1350695,14803926,23.70,666104,591189,622491,637500,633557,657941,676143,731607,747780,731574,738853,772263,838067,9045069,23.04
1,b. office ride,2704099,2063373,2296376,2869861,2664161,3140224,3094706,3411848,3562212,3148195,3160640,3430583,3252421,38798699,62.11,1758228,1478582,1610594,1827139,1749136,1897975,1959193,2025764,2044328,2007511,1996628,2051352,2038312,24444742,62.26
2,c. transit ride,635823,541390,598391,637674,639544,684019,681156,756464,782591,701350,711750,748576,742450,8861178,14.19,418068,386909,412426,422309,420330,434409,444844,470809,476018,464721,464198,472664,484270,5771975,14.70


In [7]:
cx_data15.to_clipboard(index=False)

In [5]:
cx_data16 = f'''
    with base as (
    
        select
            customer_id,
            segment,
            -- total_net_orders,
            -- office_orders,
        /*
        case 
        when office_orders = '0' then 'a. 0 office ride'
        when office_orders = '1' then 'b. 1 office ride'
        when office_orders >= '2' and office_orders <= '3' then 'c. 2-3 office'
        when office_orders >= '4' and office_orders <= '6' then 'd. 4-6 office'
        when office_orders >= '7' and office_orders <= '10' then 'e. 7-10 office'
        when office_orders > '10' then 'f. 10+ office'
        else 'a. 0 office ride'
        end office_bin
        
        case 
        when office_orders = '0' and transit_orders = '0' then 'a. no-office-transit'
        when office_orders = '1' then 'b. 1 office ride'
        when office_orders >= '2' and office_orders <= '3' then 'c. 2-3 office'
        when office_orders >= '4' and office_orders <= '6' then 'd. 4-6 office'
        when office_orders >= '7' and office_orders <= '10' then 'e. 7-10 office'
        when office_orders > '10' then 'f. 10+ office'
        when transit_orders = '1' then 'g. 1 transit ride'
        when transit_orders >= '2' and transit_orders <= '3' then 'h. 2-3 transit ride'
        when transit_orders > '3' then 'i. 3+ transit ride'
        else 'a. no-office-transit'
        end office_bin
        */
        
        case 
        when office_orders = '0' and transit_orders = '0' then 'a. no-office-transit'
        when office_orders >= '1' then 'b. office ride'
        when transit_orders >= '1' then 'c. transit ride'
        else 'a. no-office-transit'
        end office_bin
        
        
    from 
        hive.experiments_internal.customer_base_office_transit_tag_amj2025
    ),
    
    orders as (
    
    select
        date_format(date_trunc('week', date_parse(yyyymmdd, '%Y%m%d')), '%Y%m%d') week_start_date,
        customer_id,
        count(distinct order_id) gross_orders,
        count(distinct case when order_status = 'dropped' then order_id end) net_orders
    from 
        orders.order_logs_fact
    where
        yyyymmdd >= '20241216'
        and yyyymmdd <= '20250316'
        and city_name = 'Bangalore'
        and service_obj_service_name in ('Auto','Auto Lite','Auto Pool','AutoPremium','CityAuto','Link','Bike Lite','Bike Pink','CabEconomy','Bike Metro', 'CabPremium')
    group by 1,2
    )
    
    
    select
        segment,
        office_bin,
        sum(case when week_start_date = '20241216' then gross_orders end ) w01_gross,
        sum(case when week_start_date = '20241223' then gross_orders end ) w02_gross,
        sum(case when week_start_date = '20241230' then gross_orders end ) w03_gross,
        sum(case when week_start_date = '20250106' then gross_orders end ) w04_gross,
        sum(case when week_start_date = '20250113' then gross_orders end ) w05_gross,
        sum(case when week_start_date = '20250120' then gross_orders end ) w06_gross,
        sum(case when week_start_date = '20250127' then gross_orders end ) w07_gross,
        sum(case when week_start_date = '20250203' then gross_orders end ) w08_gross,
        sum(case when week_start_date = '20250210' then gross_orders end ) w09_gross,
        sum(case when week_start_date = '20250217' then gross_orders end ) w10_gross,
        sum(case when week_start_date = '20250224' then gross_orders end ) w11_gross,
        sum(case when week_start_date = '20250303' then gross_orders end ) w12_gross,
        sum(case when week_start_date = '20250310' then gross_orders end ) w13_gross,
        
        sum(orders.gross_orders) as overall_gross,
        100.00*sum(gross_orders)/sum(sum(gross_orders))over() as pct,
        
        sum(case when week_start_date = '20241216' then net_orders end ) w01_net,
        sum(case when week_start_date = '20241223' then net_orders end ) w02_net,
        sum(case when week_start_date = '20241230' then net_orders end ) w03_net,
        sum(case when week_start_date = '20250106' then net_orders end ) w04_net,
        sum(case when week_start_date = '20250113' then net_orders end ) w05_net,
        sum(case when week_start_date = '20250120' then net_orders end ) w06_net,
        sum(case when week_start_date = '20250127' then net_orders end ) w07_net,
        sum(case when week_start_date = '20250203' then net_orders end ) w08_net,
        sum(case when week_start_date = '20250210' then net_orders end ) w09_net,
        sum(case when week_start_date = '20250217' then net_orders end ) w10_net,
        sum(case when week_start_date = '20250224' then net_orders end ) w11_net,
        sum(case when week_start_date = '20250303' then net_orders end ) w12_net,
        sum(case when week_start_date = '20250310' then net_orders end ) w13_net,
        
        sum(orders.net_orders) as overall_gross,
        100.00*sum(net_orders)/sum(sum(net_orders))over() as pct

    from
        base
    left join 
        orders
        on base.customer_id = orders.customer_id
    group by 1,2
    order by 1,2
       
'''
cx_data16 = pd.read_sql(cx_data16, conn3)
cx_data16.head()

,segment,office_bin,w01_gross,w02_gross,w03_gross,w04_gross,w05_gross,w06_gross,w07_gross,w08_gross,w09_gross,w10_gross,w11_gross,w12_gross,w13_gross,overall_gross,pct,w01_net,w02_net,w03_net,w04_net,w05_net,w06_net,w07_net,w08_net,w09_net,w10_net,w11_net,w12_net,w13_net,overall_gross,pct
0,1-DAILY,a. no-office-transit,85815,61744,72661,96535,90346,106859,104604,110044,111764,100116,96383,102946,95000,1234817,1.98,56001,44196,50935,61968,57859,65799,67169,66193,64992,64546,61062,63370,60158,784248,2.00
1,1-DAILY,b. office ride,806344,603605,677696,910344,833468,984865,964268,1031829,1057744,935267,923142,982890,894200,11605662,18.58,530036,439153,485220,577133,548643,595295,614452,610952,603900,598079,583201,586357,559831,7332252,18.68
2,1-DAILY,c. transit ride,86144,66955,77910,97419,91459,106030,103776,110210,111995,101203,97743,103911,97037,1251792,2.00,57516,48813,55335,63853,59944,67044,67800,68113,67147,66930,63224,64900,62586,813205,2.07
3,2-WEEKLY,a. no-office-transit,140647,110569,123338,147285,141163,156558,152257,166182,169367,150812,147996,154745,149910,1910829,3.06,87620,75788,83135,92326,88837,93595,94070,96486,94959,93507,90119,92044,91723,1174209,2.99
4,2-WEEKLY,b. office ride,758919,581516,653218,824932,756374,882601,859922,932532,967540,845658,842091,900429,843282,10649014,17.05,485476,411853,456178,517320,491144,523248,533188,542080,543640,527052,520774,527796,516802,6596551,16.80


In [6]:
cx_data16

,segment,office_bin,w01_gross,w02_gross,w03_gross,w04_gross,w05_gross,w06_gross,w07_gross,w08_gross,w09_gross,w10_gross,w11_gross,w12_gross,w13_gross,overall_gross,pct,w01_net,w02_net,w03_net,w04_net,w05_net,w06_net,w07_net,w08_net,w09_net,w10_net,w11_net,w12_net,w13_net,overall_gross,pct
0,1-DAILY,a. no-office-transit,85815,61744,72661,96535,90346,106859,104604,110044,111764,100116,96383,102946,95000,1234817,1.98,56001,44196,50935,61968,57859,65799,67169,66193,64992,64546,61062,63370,60158,784248,2.00
1,1-DAILY,b. office ride,806344,603605,677696,910344,833468,984865,964268,1031829,1057744,935267,923142,982890,894200,11605662,18.58,530036,439153,485220,577133,548643,595295,614452,610952,603900,598079,583201,586357,559831,7332252,18.68
2,1-DAILY,c. transit ride,86144,66955,77910,97419,91459,106030,103776,110210,111995,101203,97743,103911,97037,1251792,2.00,57516,48813,55335,63853,59944,67044,67800,68113,67147,66930,63224,64900,62586,813205,2.07
3,2-WEEKLY,a. no-office-transit,140647,110569,123338,147285,141163,156558,152257,166182,169367,150812,147996,154745,149910,1910829,3.06,87620,75788,83135,92326,88837,93595,94070,96486,94959,93507,90119,92044,91723,1174209,2.99
4,2-WEEKLY,b. office ride,758919,581516,653218,824932,756374,882601,859922,932532,967540,845658,842091,900429,843282,10649014,17.05,485476,411853,456178,517320,491144,523248,533188,542080,543640,527052,520774,527796,516802,6596551,16.80
5,2-WEEKLY,c. transit ride,130191,109366,124835,138756,135243,146280,144846,155101,159436,140453,141689,146653,140433,1813282,2.90,84504,77685,86181,90179,87507,91111,92368,94287,94462,90862,89935,90016,89140,1158237,2.95
6,3-MONTHLY,a. no-office-transit,595706,494568,543534,595880,610153,645838,643367,760411,741312,610153,609554,631154,648507,8130137,13.02,358892,326781,348897,367007,373754,375443,387581,437591,411823,374560,369695,370305,394554,4896883,12.47
7,3-MONTHLY,b. office ride,977681,751195,837618,1012808,968204,1138599,1137197,1309833,1330112,1125928,1124868,1216876,1183811,14114730,22.60,633594,534874,578436,651091,636055,692535,722139,784698,767180,718988,712779,730022,744356,8906747,22.69
8,3-MONTHLY,c. transit ride,339233,294595,327135,342183,356878,368435,369602,426252,420832,357361,361261,371979,370897,4706643,7.53,220745,208883,222549,226827,233323,233391,240888,264747,253721,234903,234218,233353,240279,3047827,7.76
9,4-OTHER,a. no-office-transit,243449,196096,193917,163871,154787,177080,175782,189271,261170,278135,311006,366532,420723,3131819,5.01,145410,127083,121068,99874,97128,106393,109414,112151,156317,179435,198261,225662,268703,1946899,4.96


In [8]:
cx_data16.to_clipboard(index=False)

In [15]:
cx_data01 = f'''
    with base as (
    select 
        customer_id, 
        segment  
    from 
        hive.experiments_internal.customer_segment_amj25
    ),
    
    orders as (
    
    select
        date_format(date_trunc('week', date_parse(yyyymmdd, '%Y%m%d')), '%Y%m%d') week_start_date,
        customer_id,
        count(distinct order_id) gross_orders,
        count(distinct case when order_status = 'dropped' then order_id end) net_orders
    from 
        orders.order_logs_fact
    where
        yyyymmdd >= '20241216'
        and yyyymmdd <= '20250316'
        and city_name = 'Bangalore'
        and service_obj_service_name in ('Auto','Auto Lite','Auto Pool','AutoPremium','CityAuto','Link','Bike Lite','Bike Pink','CabEconomy','Bike Metro', 'CabPremium')
    group by 1,2
    ),
    
    appo as (
    
    select
        userid,
            /*case 
                when regexp_like(apps_installed, 'jira|miro|zoho|teams||intune|figma|slack|confluence|webex|github') = true 
                then 'office-apps' 
            else 'no-office-apps' end office_tag,  */
        
        CASE 
            WHEN apps_installed LIKE '%uber%' OR apps_installed LIKE '%ola%' OR apps_installed LIKE '%blusmart%' OR apps_installed LIKE '%nammayatri%' OR apps_installed LIKE '%in drive%' 
            OR apps_installed LIKE '%quick ride%' OR apps_installed LIKE '%jugnoo%' 
        THEN 'RHA' 
            else 'NRHA' end as RHA_Check

        /*CASE 
            when apps_installed LIKE '%zomato%' OR apps_installed LIKE '%swiggy%' 
        THEN 'Zwigato' 
            else 'Non-Zwigato' end as Zwigato_check*/

            
    from 
        datasets.customer_apps_installed_snapshot
    )
    
    
    select
        segment,
        coalesce(RHA_Check, 'UNKNOWN') RHA_Check,
        sum(case when week_start_date = '20241216' then gross_orders end ) w01_gross,
        sum(case when week_start_date = '20241223' then gross_orders end ) w02_gross,
        sum(case when week_start_date = '20241230' then gross_orders end ) w03_gross,
        sum(case when week_start_date = '20250106' then gross_orders end ) w04_gross,
        sum(case when week_start_date = '20250113' then gross_orders end ) w05_gross,
        sum(case when week_start_date = '20250120' then gross_orders end ) w06_gross,
        sum(case when week_start_date = '20250127' then gross_orders end ) w07_gross,
        sum(case when week_start_date = '20250203' then gross_orders end ) w08_gross,
        sum(case when week_start_date = '20250210' then gross_orders end ) w09_gross,
        sum(case when week_start_date = '20250217' then gross_orders end ) w10_gross,
        sum(case when week_start_date = '20250224' then gross_orders end ) w11_gross,
        sum(case when week_start_date = '20250303' then gross_orders end ) w12_gross,
        sum(case when week_start_date = '20250310' then gross_orders end ) w13_gross,
        
        sum(orders.gross_orders) as overall_gross,
        100.00*sum(gross_orders)/sum(sum(gross_orders))over() as pct,
        
        sum(case when week_start_date = '20241216' then net_orders end ) w01_net,
        sum(case when week_start_date = '20241223' then net_orders end ) w02_net,
        sum(case when week_start_date = '20241230' then net_orders end ) w03_net,
        sum(case when week_start_date = '20250106' then net_orders end ) w04_net,
        sum(case when week_start_date = '20250113' then net_orders end ) w05_net,
        sum(case when week_start_date = '20250120' then net_orders end ) w06_net,
        sum(case when week_start_date = '20250127' then net_orders end ) w07_net,
        sum(case when week_start_date = '20250203' then net_orders end ) w08_net,
        sum(case when week_start_date = '20250210' then net_orders end ) w09_net,
        sum(case when week_start_date = '20250217' then net_orders end ) w10_net,
        sum(case when week_start_date = '20250224' then net_orders end ) w11_net,
        sum(case when week_start_date = '20250303' then net_orders end ) w12_net,
        sum(case when week_start_date = '20250310' then net_orders end ) w13_net,
        
        sum(orders.net_orders) as overall_gross,
        100.00*sum(net_orders)/sum(sum(net_orders))over() as pct
      
    from
        base
    left join 
        orders
        on base.customer_id = orders.customer_id
    left join
        appo
        on base.customer_id = appo.userid
    group by 1,2
    order by 1,2 
'''
cx_data11 = pd.read_sql(cx_data01, conn1)
cx_data11.head()

,segment,RHA_Check,w01_gross,w02_gross,w03_gross,w04_gross,w05_gross,w06_gross,w07_gross,w08_gross,w09_gross,w10_gross,w11_gross,w12_gross,w13_gross,overall_gross,pct,w01_net,w02_net,w03_net,w04_net,w05_net,w06_net,w07_net,w08_net,w09_net,w10_net,w11_net,w12_net,w13_net,overall_gross,pct
0,1-DAILY,NRHA,62796,54607,61135,70769,68221,75441,73977,78978,81685,72816,71882,74706,69023,916036,1.47,43435,40966,44425,47932,46199,48847,49876,50168,50305,49495,47858,47966,46081,613553,1.56
1,1-DAILY,RHA,319145,249438,282990,353098,324677,376262,367977,390780,398452,355609,352433,370688,340591,4482140,7.18,213359,183969,203896,228988,215861,233257,238702,236545,233255,232616,226192,226738,218887,2892265,7.37
2,1-DAILY,UNKNOWN,143507,106138,118190,160005,146149,169047,167437,175550,180569,161992,160483,169408,156570,2015045,3.23,95426,78157,86216,102810,98148,103908,108143,105962,105096,105462,103446,103345,99087,1295206,3.30
3,2-WEEKLY,NRHA,197621,171431,192919,218720,210282,230720,227441,245750,251781,221870,219622,229558,216165,2833880,4.54,130978,123475,135182,144876,138939,144954,147457,150604,150714,145884,141398,142685,139058,1836204,4.68
4,2-WEEKLY,RHA,885261,668955,756295,967509,890082,1038914,1011231,1094943,1127670,991177,982143,1047562,971945,12433687,19.91,563960,471863,525460,603349,571767,616316,626630,635698,632173,616935,605361,613577,597579,7680668,19.56


In [16]:
cx_data11

,segment,RHA_Check,w01_gross,w02_gross,w03_gross,w04_gross,w05_gross,w06_gross,w07_gross,w08_gross,w09_gross,w10_gross,w11_gross,w12_gross,w13_gross,overall_gross,pct,w01_net,w02_net,w03_net,w04_net,w05_net,w06_net,w07_net,w08_net,w09_net,w10_net,w11_net,w12_net,w13_net,overall_gross,pct
0,1-DAILY,NRHA,62796,54607,61135,70769,68221,75441,73977,78978,81685,72816,71882,74706,69023,916036,1.47,43435,40966,44425,47932,46199,48847,49876,50168,50305,49495,47858,47966,46081,613553,1.56
1,1-DAILY,RHA,319145,249438,282990,353098,324677,376262,367977,390780,398452,355609,352433,370688,340591,4482140,7.18,213359,183969,203896,228988,215861,233257,238702,236545,233255,232616,226192,226738,218887,2892265,7.37
2,1-DAILY,UNKNOWN,143507,106138,118190,160005,146149,169047,167437,175550,180569,161992,160483,169408,156570,2015045,3.23,95426,78157,86216,102810,98148,103908,108143,105962,105096,105462,103446,103345,99087,1295206,3.30
3,2-WEEKLY,NRHA,197621,171431,192919,218720,210282,230720,227441,245750,251781,221870,219622,229558,216165,2833880,4.54,130978,123475,135182,144876,138939,144954,147457,150604,150714,145884,141398,142685,139058,1836204,4.68
4,2-WEEKLY,RHA,885261,668955,756295,967509,890082,1038914,1011231,1094943,1127670,991177,982143,1047562,971945,12433687,19.91,563960,471863,525460,603349,571767,616316,626630,635698,632173,616935,605361,613577,597579,7680668,19.56
5,2-WEEKLY,UNKNOWN,385365,271338,305336,428036,387246,461248,448444,484202,503014,441207,433653,469716,438236,5457041,8.74,244684,190712,213138,264017,249344,269937,277083,278096,277878,271587,265541,271984,262330,3336331,8.50
6,3-MONTHLY,NRHA,430050,382115,428973,464098,477900,488239,488016,571638,563494,466460,467445,482938,493598,6204964,9.93,278174,269134,290240,306550,311718,305334,314792,352564,335502,304852,302123,301452,317520,3989955,10.16
7,3-MONTHLY,RHA,1113238,892456,982005,1117884,1110028,1255306,1252006,1447100,1445438,1217190,1224600,1298156,1275440,15630847,25.02,702580,616352,657292,706374,707780,752412,780817,854893,825173,768211,765705,774469,796712,9708770,24.73
8,3-MONTHLY,UNKNOWN,383697,277635,310102,386023,368703,440888,443310,513453,517999,438630,432466,468851,461509,5443266,8.71,241788,193398,211017,242808,237310,262496,276155,300617,291728,274385,267383,275947,282175,3357207,8.55
9,4-OTHER,NRHA,132965,112717,111916,95647,89191,100260,99165,103159,150834,170469,190522,225139,249110,1831094,2.93,85742,78361,74460,63071,60109,64727,66126,66296,97265,116703,128350,146335,166314,1213859,3.09


In [19]:
cx_data11.to_clipboard(index=False)

In [17]:
cx_data02 = f'''
    with base as (
    select 
        customer_id, 
        segment  
    from 
        hive.experiments_internal.customer_segment_amj25
    ),
    
    orders as (
    
    select
        date_format(date_trunc('week', date_parse(yyyymmdd, '%Y%m%d')), '%Y%m%d') week_start_date,
        customer_id,
        count(distinct order_id) gross_orders,
        count(distinct case when order_status = 'dropped' then order_id end) net_orders
    from 
        orders.order_logs_fact
    where
        yyyymmdd >= '20241216'
        and yyyymmdd <= '20250316'
        and city_name = 'Bangalore'
        and service_obj_service_name in ('Auto','Auto Lite','Auto Pool','AutoPremium','CityAuto','Link','Bike Lite','Bike Pink','CabEconomy','Bike Metro', 'CabPremium')
    group by 1,2
    ),
    
    appo as (
    
    select
        userid,
            /* 
            case 
                when regexp_like(apps_installed, 'jira|miro|zoho|teams||intune|figma|slack|confluence|webex|github') = true 
                then 'office-apps' 
            else 'no-office-apps' end office_tag,  
        
        CASE 
            WHEN apps_installed LIKE '%uber%' OR apps_installed LIKE '%ola%' OR apps_installed LIKE '%blusmart%' OR apps_installed LIKE '%nammayatri%' OR apps_installed LIKE '%in drive%' 
            OR apps_installed LIKE '%quick ride%' OR apps_installed LIKE '%jugnoo%' 
        THEN 'RHA' 
            else 'NRHA' end as RHA_Check,
        */

        CASE 
            when apps_installed LIKE '%zomato%' OR apps_installed LIKE '%swiggy%' 
        THEN 'Zwigato' 
            else 'Non-Zwigato' end as Zwigato_check

            
    from 
        datasets.customer_apps_installed_snapshot
    )
    
    
    select
        segment,
        coalesce(Zwigato_check, 'UNKNOWN') Zwigato_check,
        sum(case when week_start_date = '20241216' then gross_orders end ) w01_gross,
        sum(case when week_start_date = '20241223' then gross_orders end ) w02_gross,
        sum(case when week_start_date = '20241230' then gross_orders end ) w03_gross,
        sum(case when week_start_date = '20250106' then gross_orders end ) w04_gross,
        sum(case when week_start_date = '20250113' then gross_orders end ) w05_gross,
        sum(case when week_start_date = '20250120' then gross_orders end ) w06_gross,
        sum(case when week_start_date = '20250127' then gross_orders end ) w07_gross,
        sum(case when week_start_date = '20250203' then gross_orders end ) w08_gross,
        sum(case when week_start_date = '20250210' then gross_orders end ) w09_gross,
        sum(case when week_start_date = '20250217' then gross_orders end ) w10_gross,
        sum(case when week_start_date = '20250224' then gross_orders end ) w11_gross,
        sum(case when week_start_date = '20250303' then gross_orders end ) w12_gross,
        sum(case when week_start_date = '20250310' then gross_orders end ) w13_gross,
        
        sum(orders.gross_orders) as overall_gross,
        100.00*sum(gross_orders)/sum(sum(gross_orders))over() as pct,
        
        sum(case when week_start_date = '20241216' then net_orders end ) w01_net,
        sum(case when week_start_date = '20241223' then net_orders end ) w02_net,
        sum(case when week_start_date = '20241230' then net_orders end ) w03_net,
        sum(case when week_start_date = '20250106' then net_orders end ) w04_net,
        sum(case when week_start_date = '20250113' then net_orders end ) w05_net,
        sum(case when week_start_date = '20250120' then net_orders end ) w06_net,
        sum(case when week_start_date = '20250127' then net_orders end ) w07_net,
        sum(case when week_start_date = '20250203' then net_orders end ) w08_net,
        sum(case when week_start_date = '20250210' then net_orders end ) w09_net,
        sum(case when week_start_date = '20250217' then net_orders end ) w10_net,
        sum(case when week_start_date = '20250224' then net_orders end ) w11_net,
        sum(case when week_start_date = '20250303' then net_orders end ) w12_net,
        sum(case when week_start_date = '20250310' then net_orders end ) w13_net,
        
        sum(orders.net_orders) as overall_gross,
        100.00*sum(net_orders)/sum(sum(net_orders))over() as pct

    from
        base
    left join 
        orders
        on base.customer_id = orders.customer_id
    left join
        appo
        on base.customer_id = appo.userid
    group by 1,2
    order by 1,2
'''
cx_data12 = pd.read_sql(cx_data02, conn1)
cx_data12.head()

,segment,Zwigato_check,w01_gross,w02_gross,w03_gross,w04_gross,w05_gross,w06_gross,w07_gross,w08_gross,w09_gross,w10_gross,w11_gross,w12_gross,w13_gross,overall_gross,pct,w01_net,w02_net,w03_net,w04_net,w05_net,w06_net,w07_net,w08_net,w09_net,w10_net,w11_net,w12_net,w13_net,overall_gross,pct
0,1-DAILY,Non-Zwigato,81265,70487,78228,90873,86702,97732,96145,101465,103863,92928,91658,94748,87363,1173457,1.88,54715,51736,55616,60228,57394,61772,63119,63102,62506,62040,59676,59232,56850,767986,1.96
1,1-DAILY,UNKNOWN,143507,106138,118190,160005,146149,169047,167437,175550,180569,161992,160483,169408,156570,2015045,3.23,95426,78157,86216,102810,98148,103908,108143,105962,105096,105462,103446,103345,99087,1295206,3.30
2,1-DAILY,Zwigato,300676,233558,265897,332994,306196,353971,345809,368293,376274,335497,332657,350646,322251,4224719,6.76,202079,173199,192705,216692,204666,220332,225459,223611,221054,220071,214374,215472,208118,2737832,6.97
3,2-WEEKLY,Non-Zwigato,242767,207267,231336,265971,255191,281840,278139,299522,307042,271010,266425,278637,260341,3445488,5.52,157587,146771,159508,172755,165407,174601,177703,180962,180375,175053,168762,169455,164385,2193324,5.59
4,2-WEEKLY,UNKNOWN,385365,271338,305336,428036,387246,461248,448444,484202,503014,441207,433653,469716,438236,5457041,8.74,244684,190712,213138,264017,249344,269937,277083,278096,277878,271587,265541,271984,262330,3336331,8.50


In [18]:
cx_data12

,segment,Zwigato_check,w01_gross,w02_gross,w03_gross,w04_gross,w05_gross,w06_gross,w07_gross,w08_gross,w09_gross,w10_gross,w11_gross,w12_gross,w13_gross,overall_gross,pct,w01_net,w02_net,w03_net,w04_net,w05_net,w06_net,w07_net,w08_net,w09_net,w10_net,w11_net,w12_net,w13_net,overall_gross,pct
0,1-DAILY,Non-Zwigato,81265,70487,78228,90873,86702,97732,96145,101465,103863,92928,91658,94748,87363,1173457,1.88,54715,51736,55616,60228,57394,61772,63119,63102,62506,62040,59676,59232,56850,767986,1.96
1,1-DAILY,UNKNOWN,143507,106138,118190,160005,146149,169047,167437,175550,180569,161992,160483,169408,156570,2015045,3.23,95426,78157,86216,102810,98148,103908,108143,105962,105096,105462,103446,103345,99087,1295206,3.30
2,1-DAILY,Zwigato,300676,233558,265897,332994,306196,353971,345809,368293,376274,335497,332657,350646,322251,4224719,6.76,202079,173199,192705,216692,204666,220332,225459,223611,221054,220071,214374,215472,208118,2737832,6.97
3,2-WEEKLY,Non-Zwigato,242767,207267,231336,265971,255191,281840,278139,299522,307042,271010,266425,278637,260341,3445488,5.52,157587,146771,159508,172755,165407,174601,177703,180962,180375,175053,168762,169455,164385,2193324,5.59
4,2-WEEKLY,UNKNOWN,385365,271338,305336,428036,387246,461248,448444,484202,503014,441207,433653,469716,438236,5457041,8.74,244684,190712,213138,264017,249344,269937,277083,278096,277878,271587,265541,271984,262330,3336331,8.50
5,2-WEEKLY,Zwigato,840115,633119,717878,920258,845173,987794,960533,1041171,1072409,942037,935340,998483,927769,11822079,18.93,537351,448567,501134,575470,545299,586669,596384,605340,602512,587766,577997,586807,572252,7323548,18.65
6,3-MONTHLY,Non-Zwigato,514458,458765,504357,549648,569018,588528,587566,681705,669957,555401,547607,567185,573466,7367661,11.80,327950,316249,334932,356064,364065,362188,372820,414246,392940,356443,348802,347532,363362,4657593,11.86
7,3-MONTHLY,UNKNOWN,383697,277635,310102,386023,368703,440888,443310,513453,517999,438630,432466,468851,461509,5443266,8.71,241788,193398,211017,242808,237310,262496,276155,300617,291728,274385,267383,275947,282175,3357207,8.55
8,3-MONTHLY,Zwigato,1028830,815806,906621,1032334,1018910,1155017,1152456,1337033,1338975,1128249,1144438,1213909,1195572,14468150,23.16,652804,569237,612600,656860,655433,695558,722789,793211,767735,716620,719026,728389,750870,9041132,23.03
9,4-OTHER,Non-Zwigato,164871,140220,133739,113695,106070,120745,118981,122806,178146,202342,225652,263981,292482,2183730,3.50,104599,95229,87306,73338,70737,77427,78561,77839,113819,136872,150202,169375,193174,1428478,3.64


In [20]:
cx_data12.to_clipboard(index=False)

In [21]:
cx_data03 = f'''
    with base as (
    select 
        customer_id, 
        segment  
    from 
        hive.experiments_internal.customer_segment_amj25
    ),
    
    orders as (
    
    select
        date_format(date_trunc('week', date_parse(yyyymmdd, '%Y%m%d')), '%Y%m%d') week_start_date,
        customer_id,
        count(distinct order_id) gross_orders,
        count(distinct case when order_status = 'dropped' then order_id end) net_orders
    from 
        orders.order_logs_fact
    where
        yyyymmdd >= '20241216'
        and yyyymmdd <= '20250316'
        and city_name = 'Bangalore'
        and service_obj_service_name in ('Auto','Auto Lite','Auto Pool','AutoPremium','CityAuto','Link','Bike Lite','Bike Pink','CabEconomy','Bike Metro', 'CabPremium')
    group by 1,2
    ),
    
    appo as (
    
    select
        userid,
            /* 
            case 
                when regexp_like(apps_installed, 'jira|miro|zoho|teams||intune|figma|slack|confluence|webex|github') = true 
                then 'office-apps' 
            else 'no-office-apps' end office_tag,  
        */
        CASE 
            WHEN apps_installed LIKE '%uber%' OR apps_installed LIKE '%ola%' OR apps_installed LIKE '%blusmart%' OR apps_installed LIKE '%nammayatri%' OR apps_installed LIKE '%in drive%' 
            OR apps_installed LIKE '%quick ride%' OR apps_installed LIKE '%jugnoo%' 
        THEN 'RHA' 
            else 'NRHA' end as RHA_Check,

        CASE 
            when apps_installed LIKE '%zomato%' OR apps_installed LIKE '%swiggy%' 
        THEN 'Zwigato' 
            else 'Non-Zwigato' end as Zwigato_check

            
    from 
        datasets.customer_apps_installed_snapshot
    )
    
    
    select
        segment,
        
        case 
        when RHA_Check = 'RHA' and Zwigato_check = 'Zwigato' then '1. BOTH'
        when RHA_Check = 'NRHA' and Zwigato_check = 'Zwigato' then '2. ONLY Zwigato'
        when RHA_Check = 'RHA' and Zwigato_check = 'Non-Zwigato' then '3. ONLY RHA'
        when RHA_Check = 'NRHA' and Zwigato_check = 'Non-Zwigato' then '4. NONE'
        ELSE '5. IOS' end tech_savvy,
        sum(case when week_start_date = '20241216' then gross_orders end ) w01_gross,
        sum(case when week_start_date = '20241223' then gross_orders end ) w02_gross,
        sum(case when week_start_date = '20241230' then gross_orders end ) w03_gross,
        sum(case when week_start_date = '20250106' then gross_orders end ) w04_gross,
        sum(case when week_start_date = '20250113' then gross_orders end ) w05_gross,
        sum(case when week_start_date = '20250120' then gross_orders end ) w06_gross,
        sum(case when week_start_date = '20250127' then gross_orders end ) w07_gross,
        sum(case when week_start_date = '20250203' then gross_orders end ) w08_gross,
        sum(case when week_start_date = '20250210' then gross_orders end ) w09_gross,
        sum(case when week_start_date = '20250217' then gross_orders end ) w10_gross,
        sum(case when week_start_date = '20250224' then gross_orders end ) w11_gross,
        sum(case when week_start_date = '20250303' then gross_orders end ) w12_gross,
        sum(case when week_start_date = '20250310' then gross_orders end ) w13_gross,
        
        sum(orders.gross_orders) as overall_gross,
        100.00*sum(gross_orders)/sum(sum(gross_orders))over() as pct,
        
        sum(case when week_start_date = '20241216' then net_orders end ) w01_net,
        sum(case when week_start_date = '20241223' then net_orders end ) w02_net,
        sum(case when week_start_date = '20241230' then net_orders end ) w03_net,
        sum(case when week_start_date = '20250106' then net_orders end ) w04_net,
        sum(case when week_start_date = '20250113' then net_orders end ) w05_net,
        sum(case when week_start_date = '20250120' then net_orders end ) w06_net,
        sum(case when week_start_date = '20250127' then net_orders end ) w07_net,
        sum(case when week_start_date = '20250203' then net_orders end ) w08_net,
        sum(case when week_start_date = '20250210' then net_orders end ) w09_net,
        sum(case when week_start_date = '20250217' then net_orders end ) w10_net,
        sum(case when week_start_date = '20250224' then net_orders end ) w11_net,
        sum(case when week_start_date = '20250303' then net_orders end ) w12_net,
        sum(case when week_start_date = '20250310' then net_orders end ) w13_net,
        
        sum(orders.net_orders) as overall_gross,
        100.00*sum(net_orders)/sum(sum(net_orders))over() as pct

    from
        base
    left join 
        orders
        on base.customer_id = orders.customer_id
    left join
        appo
        on base.customer_id = appo.userid
    group by 1,2
    order by 1,2
'''
cx_data13 = pd.read_sql(cx_data03, conn1)
cx_data13.head()

,segment,tech_savvy,w01_gross,w02_gross,w03_gross,w04_gross,w05_gross,w06_gross,w07_gross,w08_gross,w09_gross,w10_gross,w11_gross,w12_gross,w13_gross,overall_gross,pct,w01_net,w02_net,w03_net,w04_net,w05_net,w06_net,w07_net,w08_net,w09_net,w10_net,w11_net,w12_net,w13_net,overall_gross,pct
0,1-DAILY,1. BOTH,270026,207568,236618,299104,273606,318113,310511,330396,337475,300427,297987,314301,288772,3784904,6.06,180753,153534,171183,193602,182388,196992,201621,199540,197117,196249,191360,192096,185769,2442204,6.22
1,1-DAILY,2. ONLY Zwigato,30650,25990,29279,33890,32590,35858,35298,37897,38799,35070,34670,36345,33479,439815,0.70,21326,19665,21522,23090,22278,23340,23838,24071,23937,23822,23014,23376,22349,295628,0.75
2,1-DAILY,3. ONLY RHA,49119,41870,46372,53994,51071,58149,57466,60384,60977,55182,54446,56387,51819,697236,1.12,32606,30435,32713,35386,33473,36265,37081,37005,36138,36367,34832,34642,33118,450061,1.15
3,1-DAILY,4. NONE,32146,28617,31856,36879,35631,39583,38679,41081,42886,37746,37212,38361,35544,476221,0.76,22109,21301,22903,24842,23921,25507,26038,26097,26368,25673,24844,24590,23732,317925,0.81
4,1-DAILY,5. IOS,143507,106138,118190,160005,146149,169047,167437,175550,180569,161992,160483,169408,156570,2015045,3.23,95426,78157,86216,102810,98148,103908,108143,105962,105096,105462,103446,103345,99087,1295206,3.30


In [22]:
cx_data13

,segment,tech_savvy,w01_gross,w02_gross,w03_gross,w04_gross,w05_gross,w06_gross,w07_gross,w08_gross,w09_gross,w10_gross,w11_gross,w12_gross,w13_gross,overall_gross,pct,w01_net,w02_net,w03_net,w04_net,w05_net,w06_net,w07_net,w08_net,w09_net,w10_net,w11_net,w12_net,w13_net,overall_gross,pct
0,1-DAILY,1. BOTH,270026,207568,236618,299104,273606,318113,310511,330396,337475,300427,297987,314301,288772,3784904,6.06,180753,153534,171183,193602,182388,196992,201621,199540,197117,196249,191360,192096,185769,2442204,6.22
1,1-DAILY,2. ONLY Zwigato,30650,25990,29279,33890,32590,35858,35298,37897,38799,35070,34670,36345,33479,439815,0.70,21326,19665,21522,23090,22278,23340,23838,24071,23937,23822,23014,23376,22349,295628,0.75
2,1-DAILY,3. ONLY RHA,49119,41870,46372,53994,51071,58149,57466,60384,60977,55182,54446,56387,51819,697236,1.12,32606,30435,32713,35386,33473,36265,37081,37005,36138,36367,34832,34642,33118,450061,1.15
3,1-DAILY,4. NONE,32146,28617,31856,36879,35631,39583,38679,41081,42886,37746,37212,38361,35544,476221,0.76,22109,21301,22903,24842,23921,25507,26038,26097,26368,25673,24844,24590,23732,317925,0.81
4,1-DAILY,5. IOS,143507,106138,118190,160005,146149,169047,167437,175550,180569,161992,160483,169408,156570,2015045,3.23,95426,78157,86216,102810,98148,103908,108143,105962,105096,105462,103446,103345,99087,1295206,3.30
5,2-WEEKLY,1. BOTH,748611,554623,628987,819109,749050,881212,855496,926768,956614,839341,833976,892082,826344,10512213,16.83,476115,391856,438500,508322,481059,519297,527951,535027,532528,519905,511999,519939,506188,6468686,16.48
6,2-WEEKLY,2. ONLY Zwigato,91504,78496,88891,101149,96123,106582,105037,114403,115795,102696,101364,106401,101425,1309866,2.10,61236,56711,62634,67148,64240,67372,68433,70313,69984,67861,65998,66868,66064,854862,2.18
7,2-WEEKLY,3. ONLY RHA,136650,114332,127308,148400,141032,157702,155735,168175,171056,151836,148167,155480,145601,1921474,3.08,87845,80007,86960,95027,90708,97019,98679,100671,99645,97030,93362,93638,91391,1211982,3.09
8,2-WEEKLY,4. NONE,106117,92935,104028,117571,114159,124138,122404,131347,135986,119174,118258,123157,114740,1524014,2.44,69742,66764,72548,77728,74699,77582,79024,80291,80730,78023,75400,75817,72994,981342,2.50
9,2-WEEKLY,5. IOS,385365,271338,305336,428036,387246,461248,448444,484202,503014,441207,433653,469716,438236,5457041,8.74,244684,190712,213138,264017,249344,269937,277083,278096,277878,271587,265541,271984,262330,3336331,8.50


In [26]:
cx_data13.to_clipboard(index=False)

In [23]:
cx_data04 = f'''
    with base as (
    select 
        customer_id, 
        segment  
    from 
        hive.experiments_internal.customer_segment_amj25
    ),
    
    orders as (
    
    select
        date_format(date_trunc('week', date_parse(yyyymmdd, '%Y%m%d')), '%Y%m%d') week_start_date,
        customer_id,
        count(distinct order_id) gross_orders,
        count(distinct case when order_status = 'dropped' then order_id end) net_orders
    from 
        orders.order_logs_fact
    where
        yyyymmdd >= '20241216'
        and yyyymmdd <= '20250316'
        and city_name = 'Bangalore'
        and service_obj_service_name in ('Auto','Auto Lite','Auto Pool','AutoPremium','CityAuto','Link','Bike Lite','Bike Pink','CabEconomy','Bike Metro', 'CabPremium')
    group by 1,2
    ),
    
    appo as (
    
    select
        userid,
            /* 
            case 
                when regexp_like(apps_installed, 'jira|miro|zoho|teams||intune|figma|slack|confluence|webex|github') = true 
                then 'office-apps' 
            else 'no-office-apps' end office_tag,  
        */
        CASE 
            WHEN apps_installed LIKE '%uber%' OR apps_installed LIKE '%ola%' OR apps_installed LIKE '%blusmart%' OR apps_installed LIKE '%nammayatri%' OR apps_installed LIKE '%in drive%' 
            OR apps_installed LIKE '%quick ride%' OR apps_installed LIKE '%jugnoo%' 
        THEN 'RHA' 
            else 'NRHA' end as RHA_Check,

        CASE 
            when apps_installed LIKE '%zomato%' OR apps_installed LIKE '%swiggy%' 
        THEN 'Zwigato' 
            else 'Non-Zwigato' end as Zwigato_check

            
    from 
        datasets.customer_apps_installed_snapshot
    )
    
    
    select
        -- segment,
        case 
        when RHA_Check = 'RHA' and Zwigato_check = 'Zwigato' then '1. BOTH'
        when RHA_Check = 'NRHA' and Zwigato_check = 'Zwigato' then '2. ONLY Zwigato'
        when RHA_Check = 'RHA' and Zwigato_check = 'Non-Zwigato' then '3. ONLY RHA'
        when RHA_Check = 'NRHA' and Zwigato_check = 'Non-Zwigato' then '4. NONE'
        ELSE '5. IOS' end tech_savvy,
        sum(case when week_start_date = '20241216' then gross_orders end ) w01_gross,
        sum(case when week_start_date = '20241223' then gross_orders end ) w02_gross,
        sum(case when week_start_date = '20241230' then gross_orders end ) w03_gross,
        sum(case when week_start_date = '20250106' then gross_orders end ) w04_gross,
        sum(case when week_start_date = '20250113' then gross_orders end ) w05_gross,
        sum(case when week_start_date = '20250120' then gross_orders end ) w06_gross,
        sum(case when week_start_date = '20250127' then gross_orders end ) w07_gross,
        sum(case when week_start_date = '20250203' then gross_orders end ) w08_gross,
        sum(case when week_start_date = '20250210' then gross_orders end ) w09_gross,
        sum(case when week_start_date = '20250217' then gross_orders end ) w10_gross,
        sum(case when week_start_date = '20250224' then gross_orders end ) w11_gross,
        sum(case when week_start_date = '20250303' then gross_orders end ) w12_gross,
        sum(case when week_start_date = '20250310' then gross_orders end ) w13_gross,
        
        sum(orders.gross_orders) as overall_gross,
        100.00*sum(gross_orders)/sum(sum(gross_orders))over() as pct,
        
        sum(case when week_start_date = '20241216' then net_orders end ) w01_net,
        sum(case when week_start_date = '20241223' then net_orders end ) w02_net,
        sum(case when week_start_date = '20241230' then net_orders end ) w03_net,
        sum(case when week_start_date = '20250106' then net_orders end ) w04_net,
        sum(case when week_start_date = '20250113' then net_orders end ) w05_net,
        sum(case when week_start_date = '20250120' then net_orders end ) w06_net,
        sum(case when week_start_date = '20250127' then net_orders end ) w07_net,
        sum(case when week_start_date = '20250203' then net_orders end ) w08_net,
        sum(case when week_start_date = '20250210' then net_orders end ) w09_net,
        sum(case when week_start_date = '20250217' then net_orders end ) w10_net,
        sum(case when week_start_date = '20250224' then net_orders end ) w11_net,
        sum(case when week_start_date = '20250303' then net_orders end ) w12_net,
        sum(case when week_start_date = '20250310' then net_orders end ) w13_net,
        
        sum(orders.net_orders) as overall_gross,
        100.00*sum(net_orders)/sum(sum(net_orders))over() as pct

    from
        base
    left join 
        orders
        on base.customer_id = orders.customer_id
    left join
        appo
        on base.customer_id = appo.userid
    group by 1
    order by 1
'''
cx_data14 = pd.read_sql(cx_data04, conn1)
cx_data14.head()

,tech_savvy,w01_gross,w02_gross,w03_gross,w04_gross,w05_gross,w06_gross,w07_gross,w08_gross,w09_gross,w10_gross,w11_gross,w12_gross,w13_gross,overall_gross,pct,w01_net,w02_net,w03_net,w04_net,w05_net,w06_net,w07_net,w08_net,w09_net,w10_net,w11_net,w12_net,w13_net,overall_gross,pct
0,1. BOTH,2089508,1596573,1773213,2127212,2004649,2336802,2300145,2555249,2663453,2350167,2384250,2573395,2480108,29234724,46.80,1332944,1124914,1220438,1339247,1293611,1398846,1438524,1500658,1515797,1485249,1489885,1532691,1551282,18224086,46.42
1,2. ONLY Zwigato,334075,287464,322906,345008,336314,362794,361033,408357,423040,377346,386267,413664,414202,4772470,7.64,221330,206384,223950,229766,223963,230130,236731,254799,257812,252364,254537,262706,272237,3126709,7.96
2,3. ONLY RHA,519226,448555,480751,520748,513108,562927,558988,620358,640665,573468,574185,612228,606467,7231674,11.58,331119,307758,320283,332713,327925,345702,354137,375008,377424,369480,365861,373703,385168,4566281,11.63
3,4. NONE,509520,451489,492232,524616,529118,552643,549194,615407,649575,576709,586101,623283,639311,7299198,11.69,330352,318431,334150,346235,346003,347094,355799,379941,391301,379738,380267,391575,413637,4714523,12.01
4,5. IOS,983178,709725,788424,1020053,943003,1123674,1111789,1227622,1285929,1141959,1138595,1246308,1205478,13925737,22.29,626655,499193,546690,638987,611521,668553,694989,717774,725792,716975,709129,735604,738325,8630187,21.98


In [24]:
cx_data14

,tech_savvy,w01_gross,w02_gross,w03_gross,w04_gross,w05_gross,w06_gross,w07_gross,w08_gross,w09_gross,w10_gross,w11_gross,w12_gross,w13_gross,overall_gross,pct,w01_net,w02_net,w03_net,w04_net,w05_net,w06_net,w07_net,w08_net,w09_net,w10_net,w11_net,w12_net,w13_net,overall_gross,pct
0,1. BOTH,2089508,1596573,1773213,2127212,2004649,2336802,2300145,2555249,2663453,2350167,2384250,2573395,2480108,29234724,46.80,1332944,1124914,1220438,1339247,1293611,1398846,1438524,1500658,1515797,1485249,1489885,1532691,1551282,18224086,46.42
1,2. ONLY Zwigato,334075,287464,322906,345008,336314,362794,361033,408357,423040,377346,386267,413664,414202,4772470,7.64,221330,206384,223950,229766,223963,230130,236731,254799,257812,252364,254537,262706,272237,3126709,7.96
2,3. ONLY RHA,519226,448555,480751,520748,513108,562927,558988,620358,640665,573468,574185,612228,606467,7231674,11.58,331119,307758,320283,332713,327925,345702,354137,375008,377424,369480,365861,373703,385168,4566281,11.63
3,4. NONE,509520,451489,492232,524616,529118,552643,549194,615407,649575,576709,586101,623283,639311,7299198,11.69,330352,318431,334150,346235,346003,347094,355799,379941,391301,379738,380267,391575,413637,4714523,12.01
4,5. IOS,983178,709725,788424,1020053,943003,1123674,1111789,1227622,1285929,1141959,1138595,1246308,1205478,13925737,22.29,626655,499193,546690,638987,611521,668553,694989,717774,725792,716975,709129,735604,738325,8630187,21.98


In [25]:
cx_data14.to_clipboard(index=False)

### regularity_segment

In [8]:
cx_data = f'''
    with base as (
    select 
        customer_id,
        taxi_regularity_segment
    from 
        datasets.iallocator_customer_segments
    where 
        run_date = '2025-03-17'
        and taxi_lifetime_last_ride_city = 'Bangalore'
    ),
    
    orders as (
    
    select
        date_format(date_trunc('week', date_parse(yyyymmdd, '%Y%m%d')), '%Y%m%d') week_start_date,
        customer_id,
        count(distinct order_id) gross_orders,
        count(distinct case when order_status = 'dropped' then order_id end) net_orders
    from 
        orders.order_logs_fact
    where
        yyyymmdd >= '20241216'
        and yyyymmdd <= '20250316'
        and city_name = 'Bangalore'
        and service_obj_service_name in ('Auto','Auto Lite','Auto Pool','AutoPremium','CityAuto','Link','Bike Lite','Bike Pink','CabEconomy','Bike Metro', 'CabPremium')
    group by 1,2
    )
    
    
    select
        taxi_regularity_segment regularity_segment,
        sum(case when week_start_date = '20241216' then gross_orders end ) w01_gross,
        sum(case when week_start_date = '20241223' then gross_orders end ) w02_gross,
        sum(case when week_start_date = '20241230' then gross_orders end ) w03_gross,
        sum(case when week_start_date = '20250106' then gross_orders end ) w04_gross,
        sum(case when week_start_date = '20250113' then gross_orders end ) w05_gross,
        sum(case when week_start_date = '20250120' then gross_orders end ) w06_gross,
        sum(case when week_start_date = '20250127' then gross_orders end ) w07_gross,
        sum(case when week_start_date = '20250203' then gross_orders end ) w08_gross,
        sum(case when week_start_date = '20250210' then gross_orders end ) w09_gross,
        sum(case when week_start_date = '20250217' then gross_orders end ) w10_gross,
        sum(case when week_start_date = '20250224' then gross_orders end ) w11_gross,
        sum(case when week_start_date = '20250303' then gross_orders end ) w12_gross,
        sum(case when week_start_date = '20250310' then gross_orders end ) w13_gross,
        
        sum(orders.gross_orders) as overall_gross,
        100.00*sum(gross_orders)/sum(sum(gross_orders))over() as pct,
        
        sum(case when week_start_date = '20241216' then net_orders end ) w01_net,
        sum(case when week_start_date = '20241223' then net_orders end ) w02_net,
        sum(case when week_start_date = '20241230' then net_orders end ) w03_net,
        sum(case when week_start_date = '20250106' then net_orders end ) w04_net,
        sum(case when week_start_date = '20250113' then net_orders end ) w05_net,
        sum(case when week_start_date = '20250120' then net_orders end ) w06_net,
        sum(case when week_start_date = '20250127' then net_orders end ) w07_net,
        sum(case when week_start_date = '20250203' then net_orders end ) w08_net,
        sum(case when week_start_date = '20250210' then net_orders end ) w09_net,
        sum(case when week_start_date = '20250217' then net_orders end ) w10_net,
        sum(case when week_start_date = '20250224' then net_orders end ) w11_net,
        sum(case when week_start_date = '20250303' then net_orders end ) w12_net,
        sum(case when week_start_date = '20250310' then net_orders end ) w13_net,
        
        sum(orders.net_orders) as overall_gross,
        100.00*sum(net_orders)/sum(sum(net_orders))over() as pct

    from
        base
    left join 
        orders
        on base.customer_id = orders.customer_id
    group by 1
       
'''
cx_data1 = pd.read_sql(cx_data, conn1)
cx_data1.head()

,regularity_segment,w01_gross,w02_gross,w03_gross,w04_gross,w05_gross,w06_gross,w07_gross,w08_gross,w09_gross,w10_gross,w11_gross,w12_gross,w13_gross,overall_gross,pct,w01_net,w02_net,w03_net,w04_net,w05_net,w06_net,w07_net,w08_net,w09_net,w10_net,w11_net,w12_net,w13_net,overall_gross,pct
0,MONTHLY,371007,330350,354032,356452,347004,355919,339255,369715,365662,304474,295943,340366,371282,4501461,7.21,237761,229851,237309,231071,226627,222571,219630,228962,221747,202208,195420,218094,245168,2916419,7.43
1,UNKNOWN,146613,134304,146907,150363,156271,171361,174054,207843,228479,209080,218319,240625,271312,2455531,3.93,85173,82210,86270,87752,92075,96283,102358,118504,128483,127044,133003,142128,167341,1448624,3.69
2,WEEKLY,1245909,1003188,1136987,1331030,1292392,1491367,1489304,1668753,1742840,1537861,1560272,1664442,1631356,18795701,30.09,794420,705913,781152,849783,838963,902318,933667,988031,1001641,974002,975882,995510,1014385,11755667,29.94
3,DAILY,1522047,1126621,1281257,1742703,1614401,1947302,1928826,2113625,2204464,1957811,1933449,2078125,1911159,23361790,37.40,989760,810897,906525,1098514,1050477,1169596,1218140,1243655,1250408,1240640,1209059,1229595,1184923,14602189,37.19
4,RARE_NEED,424854,286828,264131,237186,207636,206593,201824,221310,228567,201403,203440,210585,234186,3128543,5.01,271456,199399,177741,154690,135576,129530,130498,136368,137473,133105,132584,133210,154526,2026156,5.16


In [9]:
cx_data1

,regularity_segment,w01_gross,w02_gross,w03_gross,w04_gross,w05_gross,w06_gross,w07_gross,w08_gross,w09_gross,w10_gross,w11_gross,w12_gross,w13_gross,overall_gross,pct,w01_net,w02_net,w03_net,w04_net,w05_net,w06_net,w07_net,w08_net,w09_net,w10_net,w11_net,w12_net,w13_net,overall_gross,pct
0,MONTHLY,371007,330350,354032,356452,347004,355919,339255,369715,365662,304474,295943,340366,371282,4501461,7.21,237761,229851,237309,231071,226627,222571,219630,228962,221747,202208,195420,218094,245168,2916419,7.43
1,UNKNOWN,146613,134304,146907,150363,156271,171361,174054,207843,228479,209080,218319,240625,271312,2455531,3.93,85173,82210,86270,87752,92075,96283,102358,118504,128483,127044,133003,142128,167341,1448624,3.69
2,WEEKLY,1245909,1003188,1136987,1331030,1292392,1491367,1489304,1668753,1742840,1537861,1560272,1664442,1631356,18795701,30.09,794420,705913,781152,849783,838963,902318,933667,988031,1001641,974002,975882,995510,1014385,11755667,29.94
3,DAILY,1522047,1126621,1281257,1742703,1614401,1947302,1928826,2113625,2204464,1957811,1933449,2078125,1911159,23361790,37.40,989760,810897,906525,1098514,1050477,1169596,1218140,1243655,1250408,1240640,1209059,1229595,1184923,14602189,37.19
4,RARE_NEED,424854,286828,264131,237186,207636,206593,201824,221310,228567,201403,203440,210585,234186,3128543,5.01,271456,199399,177741,154690,135576,129530,130498,136368,137473,133105,132584,133210,154526,2026156,5.16
5,BI_WEEKLY,725083,612518,674212,719904,708491,766298,747888,845748,892652,809020,857978,934740,926274,10220806,16.36,463836,428413,456514,465139,459308,470027,475889,512661,528376,526807,553734,577747,594309,6512760,16.59


In [15]:
cx_data1.to_clipboard()

### ps_tag_link

In [82]:
cx_data2 = f'''
    with base as (
    select 
        customer_id,
        ps_tag_link
    from 
        datasets.iallocator_customer_segments
    where 
        run_date = '2025-03-17'
        and taxi_lifetime_last_ride_city = 'Bangalore'
    ),
    
    orders as (
    
    select
        date_format(date_trunc('week', date_parse(yyyymmdd, '%Y%m%d')), '%Y%m%d') week_start_date,
        customer_id,
        count(distinct order_id) gross_orders,
        count(distinct case when order_status = 'dropped' then order_id end) net_orders
    from 
        orders.order_logs_fact
    where
        yyyymmdd >= '20241216'
        and yyyymmdd <= '20250316'
        and city_name = 'Bangalore'
        and service_obj_service_name in ('Link','Bike Lite','Bike Pink','Bike Metro')
        -- and service_obj_service_name in ('Auto','Auto Lite','Auto Pool','AutoPremium','CityAuto','Link','Bike Lite','Bike Pink','CabEconomy','Bike Metro', 'CabPremium')
    group by 1,2
    )
    
    
    select
        ps_tag_link ps_tag_link,
        sum(case when week_start_date = '20241216' then gross_orders end ) w01_gross,
        sum(case when week_start_date = '20241223' then gross_orders end ) w02_gross,
        sum(case when week_start_date = '20241230' then gross_orders end ) w03_gross,
        sum(case when week_start_date = '20250106' then gross_orders end ) w04_gross,
        sum(case when week_start_date = '20250113' then gross_orders end ) w05_gross,
        sum(case when week_start_date = '20250120' then gross_orders end ) w06_gross,
        sum(case when week_start_date = '20250127' then gross_orders end ) w07_gross,
        sum(case when week_start_date = '20250203' then gross_orders end ) w08_gross,
        sum(case when week_start_date = '20250210' then gross_orders end ) w09_gross,
        sum(case when week_start_date = '20250217' then gross_orders end ) w10_gross,
        sum(case when week_start_date = '20250224' then gross_orders end ) w11_gross,
        sum(case when week_start_date = '20250303' then gross_orders end ) w12_gross,
        sum(case when week_start_date = '20250310' then gross_orders end ) w13_gross,
        
        sum(orders.gross_orders) as overall_gross,
        100.00*sum(gross_orders)/sum(sum(gross_orders))over() as pct,
        
        sum(case when week_start_date = '20241216' then net_orders end ) w01_net,
        sum(case when week_start_date = '20241223' then net_orders end ) w02_net,
        sum(case when week_start_date = '20241230' then net_orders end ) w03_net,
        sum(case when week_start_date = '20250106' then net_orders end ) w04_net,
        sum(case when week_start_date = '20250113' then net_orders end ) w05_net,
        sum(case when week_start_date = '20250120' then net_orders end ) w06_net,
        sum(case when week_start_date = '20250127' then net_orders end ) w07_net,
        sum(case when week_start_date = '20250203' then net_orders end ) w08_net,
        sum(case when week_start_date = '20250210' then net_orders end ) w09_net,
        sum(case when week_start_date = '20250217' then net_orders end ) w10_net,
        sum(case when week_start_date = '20250224' then net_orders end ) w11_net,
        sum(case when week_start_date = '20250303' then net_orders end ) w12_net,
        sum(case when week_start_date = '20250310' then net_orders end ) w13_net,
        
        sum(orders.net_orders) as overall_gross,
        100.00*sum(net_orders)/sum(sum(net_orders))over() as pct
        
    from
        base
    left join 
        orders
        on base.customer_id = orders.customer_id
    group by 1
       
'''
cx_data2 = pd.read_sql(cx_data2, conn4)
cx_data2.head()

,ps_tag_link,w01_gross,w02_gross,w03_gross,w04_gross,w05_gross,w06_gross,w07_gross,w08_gross,w09_gross,w10_gross,w11_gross,w12_gross,w13_gross,overall_gross,pct,w01_net,w02_net,w03_net,w04_net,w05_net,w06_net,w07_net,w08_net,w09_net,w10_net,w11_net,w12_net,w13_net,overall_gross,pct
0,PS,165489,126724,149801,190820,174862,195722,189447,209405,219719,184131,192356,214961,199503,2412940,12.51,95493,85435,94431,106785,98839,108295,111892,114053,115425,113973,114787,112284,112276,1383968,12.06
1,NPS,336099,269741,321300,418071,370675,412261,394780,439058,444862,368949,383321,423750,390857,4973724,25.79,200950,186185,207788,240930,216769,232863,238329,246350,243932,237578,235503,230429,230196,2947802,25.68
2,UNKNOWN,873169,643244,725046,842731,807476,922411,913131,1011590,1051622,943119,988484,1102151,1076411,11900585,61.70,519308,436601,462724,497516,474595,533636,559889,582714,590582,609548,618657,618483,643125,7147378,62.26


In [90]:
cx_data2

,ps_tag_link,w01_gross,w02_gross,w03_gross,w04_gross,w05_gross,w06_gross,w07_gross,w08_gross,w09_gross,w10_gross,w11_gross,w12_gross,w13_gross,overall_gross,pct,w01_net,w02_net,w03_net,w04_net,w05_net,w06_net,w07_net,w08_net,w09_net,w10_net,w11_net,w12_net,w13_net,overall_gross,pct
0,PS,165489,126724,149801,190820,174862,195722,189447,209405,219719,184131,192356,214961,199503,2412940,12.51,95493,85435,94431,106785,98839,108295,111892,114053,115425,113973,114787,112284,112276,1383968,12.06
1,NPS,336099,269741,321300,418071,370675,412261,394780,439058,444862,368949,383321,423750,390857,4973724,25.79,200950,186185,207788,240930,216769,232863,238329,246350,243932,237578,235503,230429,230196,2947802,25.68
2,UNKNOWN,873169,643244,725046,842731,807476,922411,913131,1011590,1051622,943119,988484,1102151,1076411,11900585,61.70,519308,436601,462724,497516,474595,533636,559889,582714,590582,609548,618657,618483,643125,7147378,62.26


In [18]:
cx_data2.to_clipboard(index=False)

### ps_tag_auto

In [84]:
cx_data3 = f'''
    with base as (
    select 
        customer_id,
        ps_tag_auto
    from 
        datasets.iallocator_customer_segments
    where 
        run_date = '2025-03-17'
        and taxi_lifetime_last_ride_city = 'Bangalore'
    ),
    
    orders as (
    
    select
        date_format(date_trunc('week', date_parse(yyyymmdd, '%Y%m%d')), '%Y%m%d') week_start_date,
        customer_id,
        count(distinct order_id) gross_orders,
        count(distinct case when order_status = 'dropped' then order_id end) net_orders
    from 
        orders.order_logs_fact
    where
        yyyymmdd >= '20241216'
        and yyyymmdd <= '20250316'
        and city_name = 'Bangalore'
        and service_obj_service_name in ('Auto','Auto Lite','Auto Pool','AutoPremium')
        -- and service_obj_service_name in ('Auto','Auto Lite','Auto Pool','AutoPremium','CityAuto','Link','Bike Lite','Bike Pink','CabEconomy','Bike Metro', 'CabPremium')
    group by 1,2
    )
    
    
    select
        ps_tag_auto ps_tag_auto,
        sum(case when week_start_date = '20241216' then gross_orders end ) w01_gross,
        sum(case when week_start_date = '20241223' then gross_orders end ) w02_gross,
        sum(case when week_start_date = '20241230' then gross_orders end ) w03_gross,
        sum(case when week_start_date = '20250106' then gross_orders end ) w04_gross,
        sum(case when week_start_date = '20250113' then gross_orders end ) w05_gross,
        sum(case when week_start_date = '20250120' then gross_orders end ) w06_gross,
        sum(case when week_start_date = '20250127' then gross_orders end ) w07_gross,
        sum(case when week_start_date = '20250203' then gross_orders end ) w08_gross,
        sum(case when week_start_date = '20250210' then gross_orders end ) w09_gross,
        sum(case when week_start_date = '20250217' then gross_orders end ) w10_gross,
        sum(case when week_start_date = '20250224' then gross_orders end ) w11_gross,
        sum(case when week_start_date = '20250303' then gross_orders end ) w12_gross,
        sum(case when week_start_date = '20250310' then gross_orders end ) w13_gross,
        
        sum(orders.gross_orders) as overall_gross,
        100.00*sum(gross_orders)/sum(sum(gross_orders))over() as pct,
        
        sum(case when week_start_date = '20241216' then net_orders end ) w01_net,
        sum(case when week_start_date = '20241223' then net_orders end ) w02_net,
        sum(case when week_start_date = '20241230' then net_orders end ) w03_net,
        sum(case when week_start_date = '20250106' then net_orders end ) w04_net,
        sum(case when week_start_date = '20250113' then net_orders end ) w05_net,
        sum(case when week_start_date = '20250120' then net_orders end ) w06_net,
        sum(case when week_start_date = '20250127' then net_orders end ) w07_net,
        sum(case when week_start_date = '20250203' then net_orders end ) w08_net,
        sum(case when week_start_date = '20250210' then net_orders end ) w09_net,
        sum(case when week_start_date = '20250217' then net_orders end ) w10_net,
        sum(case when week_start_date = '20250224' then net_orders end ) w11_net,
        sum(case when week_start_date = '20250303' then net_orders end ) w12_net,
        sum(case when week_start_date = '20250310' then net_orders end ) w13_net,
        
        sum(orders.net_orders) as overall_gross,
        100.00*sum(net_orders)/sum(sum(net_orders))over() as pct

    from
        base
    left join 
        orders
        on base.customer_id = orders.customer_id
    group by 1
       
'''
cx_data3 = pd.read_sql(cx_data3, conn4)
cx_data3.head()

,ps_tag_auto,w01_gross,w02_gross,w03_gross,w04_gross,w05_gross,w06_gross,w07_gross,w08_gross,w09_gross,w10_gross,w11_gross,w12_gross,w13_gross,overall_gross,pct,w01_net,w02_net,w03_net,w04_net,w05_net,w06_net,w07_net,w08_net,w09_net,w10_net,w11_net,w12_net,w13_net,overall_gross,pct
0,PS,700261,604129,643209,718729,682567,716753,661643,715371,725781,644904,635801,671487,651494,8772129,24.42,488470,451744,477476,491756,480636,458719,436384,447086,442864,424876,418011,432110,429845,5879977,24.82
1,NPS,1298731,992695,1085258,1353100,1251278,1433952,1365166,1478852,1513953,1344399,1328742,1407436,1349822,17203384,47.89,889683,737662,799209,898554,863873,896738,889148,910337,904906,874272,862552,890853,879323,11297110,47.69
2,UNKNOWN,544724,384027,422817,511972,510093,689510,791900,951080,1030599,952683,976850,1072176,1107027,9945458,27.69,367589,278594,302230,345652,351235,440532,518059,594273,629167,626178,637780,690304,728536,6510129,27.48


In [94]:
cx_data3

,ps_tag_auto,w01_gross,w02_gross,w03_gross,w04_gross,w05_gross,w06_gross,w07_gross,w08_gross,w09_gross,w10_gross,w11_gross,w12_gross,w13_gross,overall_gross,pct,w01_net,w02_net,w03_net,w04_net,w05_net,w06_net,w07_net,w08_net,w09_net,w10_net,w11_net,w12_net,w13_net,overall_gross,pct
0,PS,700261,604129,643209,718729,682567,716753,661643,715371,725781,644904,635801,671487,651494,8772129,24.42,488470,451744,477476,491756,480636,458719,436384,447086,442864,424876,418011,432110,429845,5879977,24.82
1,NPS,1298731,992695,1085258,1353100,1251278,1433952,1365166,1478852,1513953,1344399,1328742,1407436,1349822,17203384,47.89,889683,737662,799209,898554,863873,896738,889148,910337,904906,874272,862552,890853,879323,11297110,47.69
2,UNKNOWN,544724,384027,422817,511972,510093,689510,791900,951080,1030599,952683,976850,1072176,1107027,9945458,27.69,367589,278594,302230,345652,351235,440532,518059,594273,629167,626178,637780,690304,728536,6510129,27.48


In [13]:
cx_data3.to_clipboard(index=False)

### taxi_income_segment

In [27]:
cx_data4 = f'''
    with base as (
    select 
        customer_id,
        taxi_income_segment
    from 
        datasets.iallocator_customer_segments
    where 
        run_date = '2025-03-17'
        and taxi_lifetime_last_ride_city = 'Bangalore'
    ),
    
    orders as (
    
    select
        date_format(date_trunc('week', date_parse(yyyymmdd, '%Y%m%d')), '%Y%m%d') week_start_date,
        customer_id,
        count(distinct order_id) gross_orders,
        count(distinct case when order_status = 'dropped' then order_id end) net_orders
    from 
        orders.order_logs_fact
    where
        yyyymmdd >= '20241216'
        and yyyymmdd <= '20250316'
        and city_name = 'Bangalore'
        and service_obj_service_name in ('Auto','Auto Lite','Auto Pool','AutoPremium','CityAuto','Link','Bike Lite','Bike Pink','CabEconomy','Bike Metro', 'CabPremium')
    group by 1,2
    )
    
    
    select
        taxi_income_segment taxi_income_segment,
        sum(case when week_start_date = '20241216' then gross_orders end ) w01_gross,
        sum(case when week_start_date = '20241223' then gross_orders end ) w02_gross,
        sum(case when week_start_date = '20241230' then gross_orders end ) w03_gross,
        sum(case when week_start_date = '20250106' then gross_orders end ) w04_gross,
        sum(case when week_start_date = '20250113' then gross_orders end ) w05_gross,
        sum(case when week_start_date = '20250120' then gross_orders end ) w06_gross,
        sum(case when week_start_date = '20250127' then gross_orders end ) w07_gross,
        sum(case when week_start_date = '20250203' then gross_orders end ) w08_gross,
        sum(case when week_start_date = '20250210' then gross_orders end ) w09_gross,
        sum(case when week_start_date = '20250217' then gross_orders end ) w10_gross,
        sum(case when week_start_date = '20250224' then gross_orders end ) w11_gross,
        sum(case when week_start_date = '20250303' then gross_orders end ) w12_gross,
        sum(case when week_start_date = '20250310' then gross_orders end ) w13_gross,
        
        sum(orders.gross_orders) as overall_gross,
        100.00*sum(gross_orders)/sum(sum(gross_orders))over() as pct,
        
        sum(case when week_start_date = '20241216' then net_orders end ) w01_net,
        sum(case when week_start_date = '20241223' then net_orders end ) w02_net,
        sum(case when week_start_date = '20241230' then net_orders end ) w03_net,
        sum(case when week_start_date = '20250106' then net_orders end ) w04_net,
        sum(case when week_start_date = '20250113' then net_orders end ) w05_net,
        sum(case when week_start_date = '20250120' then net_orders end ) w06_net,
        sum(case when week_start_date = '20250127' then net_orders end ) w07_net,
        sum(case when week_start_date = '20250203' then net_orders end ) w08_net,
        sum(case when week_start_date = '20250210' then net_orders end ) w09_net,
        sum(case when week_start_date = '20250217' then net_orders end ) w10_net,
        sum(case when week_start_date = '20250224' then net_orders end ) w11_net,
        sum(case when week_start_date = '20250303' then net_orders end ) w12_net,
        sum(case when week_start_date = '20250310' then net_orders end ) w13_net,
        
        sum(orders.net_orders) as overall_gross,
        100.00*sum(net_orders)/sum(sum(net_orders))over() as pct
      
    from
        base
    left join 
        orders
        on base.customer_id = orders.customer_id
    group by 1
       
'''
cx_data4 = pd.read_sql(cx_data4, conn1)
cx_data4.head()

,taxi_income_segment,w01_gross,w02_gross,w03_gross,w04_gross,w05_gross,w06_gross,w07_gross,w08_gross,w09_gross,w10_gross,w11_gross,w12_gross,w13_gross,overall_gross,pct,w01_net,w02_net,w03_net,w04_net,w05_net,w06_net,w07_net,w08_net,w09_net,w10_net,w11_net,w12_net,w13_net,overall_gross,pct
0,HIGH_INCOME,2224651,1720441,1902615,2282992,2146313,2489955,2457526,2722572,2837937,2514670,2530699,2735837,2647786,31213994,49.97,1425044,1211912,1310382,1445645,1393608,1502188,1548466,1612807,1628498,1599188,1592357,1639872,1658363,19568330,49.84
1,UNKNOWN,1042167,795230,888157,1070478,1017123,1176582,1169955,1303273,1364976,1218172,1233608,1340510,1316506,14936737,23.91,662949,555967,606122,673657,655514,704905,732120,766911,779460,770500,772473,800198,819041,9299817,23.69
2,LOW_INCOME,172675,148309,160818,175243,174915,183973,181563,203219,215173,187021,189363,200476,204556,2397304,3.84,111003,104082,109099,114089,113162,113836,115859,124106,127662,122362,121560,123552,130582,1530954,3.90
3,MEDIUM_INCOME,996020,829829,905936,1008925,987844,1088330,1072107,1197930,1244578,1099786,1115731,1192060,1176721,13915797,22.28,643410,584722,619908,653558,640742,669396,683737,724357,732508,711756,713292,732662,752666,8862714,22.57


In [65]:
cx_data4

,taxi_income_segment,w01_gross,w02_gross,w03_gross,w04_gross,w05_gross,w06_gross,w07_gross,w08_gross,w09_gross,w10_gross,w11_gross,w12_gross,w13_gross,overall_gross,pct,w01_net,w02_net,w03_net,w04_net,w05_net,w06_net,w07_net,w08_net,w09_net,w10_net,w11_net,w12_net,w13_net,overall_gross,pct
0,HIGH_INCOME,2224651,1720441,1902615,2282992,2146313,2489955,2457526,2722572,2837937,2514670,2530699,2735837,2647786,31213994,49.97,1425044,1211912,1310382,1445645,1393608,1502188,1548466,1612807,1628498,1599188,1592357,1639872,1658363,19568330,49.84
1,UNKNOWN,1042167,795230,888157,1070478,1017123,1176582,1169955,1303273,1364976,1218172,1233608,1340510,1316506,14936737,23.91,662949,555967,606122,673657,655514,704905,732120,766911,779460,770500,772473,800198,819041,9299817,23.69
2,LOW_INCOME,172675,148309,160818,175243,174915,183973,181563,203219,215173,187021,189363,200476,204556,2397304,3.84,111003,104082,109099,114089,113162,113836,115859,124106,127662,122362,121560,123552,130582,1530954,3.90
3,MEDIUM_INCOME,996020,829829,905936,1008925,987844,1088330,1072107,1197930,1244578,1099786,1115731,1192060,1176721,13915797,22.28,643410,584722,619908,653558,640742,669396,683737,724357,732508,711756,713292,732662,752666,8862714,22.57


In [17]:
cx_data4.to_clipboard(index=False)

### taxi_service_affinity

In [35]:
cx_data5 = f'''
    with base as (
    select 
        customer_id,
        taxi_service_affinity
    from 
        datasets.iallocator_customer_segments
    where 
        run_date = '2025-03-17'
        and taxi_lifetime_last_ride_city = 'Bangalore'
    ),
    
    orders as (
    
    select
        date_format(date_trunc('week', date_parse(yyyymmdd, '%Y%m%d')), '%Y%m%d') week_start_date,
        customer_id,
        count(distinct order_id) gross_orders,
        count(distinct case when order_status = 'dropped' then order_id end) net_orders
    from 
        orders.order_logs_fact
    where
        yyyymmdd >= '20241216'
        and yyyymmdd <= '20250316'
        and city_name = 'Bangalore'
        and service_obj_service_name in ('Auto','Auto Lite','Auto Pool','AutoPremium','CityAuto','Link','Bike Lite','Bike Pink','CabEconomy','Bike Metro', 'CabPremium')
    group by 1,2
    )
    
    
    select
        taxi_service_affinity taxi_service_affinity,
        sum(case when week_start_date = '20241216' then gross_orders end ) w01_gross,
        sum(case when week_start_date = '20241223' then gross_orders end ) w02_gross,
        sum(case when week_start_date = '20241230' then gross_orders end ) w03_gross,
        sum(case when week_start_date = '20250106' then gross_orders end ) w04_gross,
        sum(case when week_start_date = '20250113' then gross_orders end ) w05_gross,
        sum(case when week_start_date = '20250120' then gross_orders end ) w06_gross,
        sum(case when week_start_date = '20250127' then gross_orders end ) w07_gross,
        sum(case when week_start_date = '20250203' then gross_orders end ) w08_gross,
        sum(case when week_start_date = '20250210' then gross_orders end ) w09_gross,
        sum(case when week_start_date = '20250217' then gross_orders end ) w10_gross,
        sum(case when week_start_date = '20250224' then gross_orders end ) w11_gross,
        sum(case when week_start_date = '20250303' then gross_orders end ) w12_gross,
        sum(case when week_start_date = '20250310' then gross_orders end ) w13_gross,
        
        sum(orders.gross_orders) as overall_gross,
        100.00*sum(gross_orders)/sum(sum(gross_orders))over() as pct,
        
        sum(case when week_start_date = '20241216' then net_orders end ) w01_net,
        sum(case when week_start_date = '20241223' then net_orders end ) w02_net,
        sum(case when week_start_date = '20241230' then net_orders end ) w03_net,
        sum(case when week_start_date = '20250106' then net_orders end ) w04_net,
        sum(case when week_start_date = '20250113' then net_orders end ) w05_net,
        sum(case when week_start_date = '20250120' then net_orders end ) w06_net,
        sum(case when week_start_date = '20250127' then net_orders end ) w07_net,
        sum(case when week_start_date = '20250203' then net_orders end ) w08_net,
        sum(case when week_start_date = '20250210' then net_orders end ) w09_net,
        sum(case when week_start_date = '20250217' then net_orders end ) w10_net,
        sum(case when week_start_date = '20250224' then net_orders end ) w11_net,
        sum(case when week_start_date = '20250303' then net_orders end ) w12_net,
        sum(case when week_start_date = '20250310' then net_orders end ) w13_net,
        
        sum(orders.net_orders) as overall_gross,
        100.00*sum(net_orders)/sum(sum(net_orders))over() as pct
    from
        base
    left join 
        orders
        on base.customer_id = orders.customer_id
    group by 1
       
'''
cx_data5 = pd.read_sql(cx_data5, conn1)
cx_data5.head()

,taxi_service_affinity,w01_gross,w02_gross,w03_gross,w04_gross,w05_gross,w06_gross,w07_gross,w08_gross,w09_gross,w10_gross,w11_gross,w12_gross,w13_gross,overall_gross,pct,w01_net,w02_net,w03_net,w04_net,w05_net,w06_net,w07_net,w08_net,w09_net,w10_net,w11_net,w12_net,w13_net,overall_gross,pct
0,ONLY_LINK,694023,549072,633749,777423,728067,820766,800881,887503,916264,778930,808812,904399,841697,10141586,16.24,425676,384096,419305,465936,443220,479210,496188,513029,514475,505866,507265,512413,506344,6173023,15.72
1,AUTO_CAB,101819,83034,86303,103516,100520,117710,116535,126968,136459,121814,117411,124327,118901,1455317,2.33,63088,56466,59397,65981,65734,70139,71919,74234,75495,73705,71965,74294,74044,896461,2.28
2,LINK_CAB,6823,6228,6975,7567,7578,8201,8106,9261,9969,8928,8679,9558,9664,107537,0.17,3975,3927,4276,4530,4610,4686,4766,5280,5519,5499,5348,5462,5680,63558,0.16
3,ONLY_AUTO,1770869,1348359,1484713,1879777,1745162,2073670,2051642,2250397,2340357,2114814,2092740,2230063,2131668,25514231,40.85,1191479,984155,1074816,1240334,1183967,1290145,1324068,1365931,1374603,1355696,1333480,1381611,1361871,16462156,41.93
4,ONLY_CAB,132408,109017,116235,144985,138696,165435,158876,170971,183356,162625,155145,163252,154834,1955835,3.13,76698,69221,74654,88489,86182,94549,95011,96028,97243,93824,90814,93927,93525,1150165,2.93


In [36]:
cx_data5

,taxi_service_affinity,w01_gross,w02_gross,w03_gross,w04_gross,w05_gross,w06_gross,w07_gross,w08_gross,w09_gross,w10_gross,w11_gross,w12_gross,w13_gross,overall_gross,pct,w01_net,w02_net,w03_net,w04_net,w05_net,w06_net,w07_net,w08_net,w09_net,w10_net,w11_net,w12_net,w13_net,overall_gross,pct
0,ONLY_LINK,694023,549072,633749,777423,728067,820766,800881,887503,916264,778930,808812,904399,841697,10141586,16.24,425676,384096,419305,465936,443220,479210,496188,513029,514475,505866,507265,512413,506344,6173023,15.72
1,AUTO_CAB,101819,83034,86303,103516,100520,117710,116535,126968,136459,121814,117411,124327,118901,1455317,2.33,63088,56466,59397,65981,65734,70139,71919,74234,75495,73705,71965,74294,74044,896461,2.28
2,LINK_CAB,6823,6228,6975,7567,7578,8201,8106,9261,9969,8928,8679,9558,9664,107537,0.17,3975,3927,4276,4530,4610,4686,4766,5280,5519,5499,5348,5462,5680,63558,0.16
3,ONLY_AUTO,1770869,1348359,1484713,1879777,1745162,2073670,2051642,2250397,2340357,2114814,2092740,2230063,2131668,25514231,40.85,1191479,984155,1074816,1240334,1183967,1290145,1324068,1365931,1374603,1355696,1333480,1381611,1361871,16462156,41.93
4,ONLY_CAB,132408,109017,116235,144985,138696,165435,158876,170971,183356,162625,155145,163252,154834,1955835,3.13,76698,69221,74654,88489,86182,94549,95011,96028,97243,93824,90814,93927,93525,1150165,2.93
5,UNKNOWN,1094444,867744,932482,933669,932769,989177,989197,1139336,1198412,1072188,1110055,1198634,1289418,13747525,22.01,695658,600209,620590,607277,605824,616808,637298,702726,724616,706126,725485,758672,846530,8847819,22.54
6,LINK_AUTO,182850,147208,169350,196292,190037,214647,210424,236752,246112,212975,219725,241395,232459,2700226,4.32,114271,103675,113950,120972,118349,126226,129909,137139,139362,134983,136298,139802,141345,1656281,4.22
7,NO_AFFINITY,452277,383147,427719,494409,483366,549234,545490,605806,631735,547375,556834,597255,566928,6841575,10.95,271561,254934,278523,293430,295140,308562,321023,333814,336815,328107,329027,330103,331313,4012352,10.22


### RHA Check

In [45]:
cx_data6 = f'''
with base as (
    select 
        customer_id,
        taxi_regularity_segment
    from 
        datasets.iallocator_customer_segments
    where 
        run_date = '2025-03-17'
        and taxi_lifetime_last_ride_city = 'Bangalore'
    ),
    
    orders as (
    
    select
        date_format(date_trunc('week', date_parse(yyyymmdd, '%Y%m%d')), '%Y%m%d') week_start_date,
        customer_id,
        count(distinct order_id) gross_orders,
        count(distinct case when order_status = 'dropped' then order_id end) net_orders
    from 
        orders.order_logs_fact
    where
        yyyymmdd >= '20241216'
        and yyyymmdd <= '20250316'
        and city_name = 'Bangalore'
        and service_obj_service_name in ('Auto','Auto Lite','Auto Pool','AutoPremium','CityAuto','Link','Bike Lite','Bike Pink','CabEconomy','Bike Metro', 'CabPremium')
    group by 1,2
    ),
    
    appo as (
    
    select
        userid,
            /*case 
                when regexp_like(apps_installed, 'jira|miro|zoho|teams||intune|figma|slack|confluence|webex|github') = true 
                then 'office-apps' 
            else 'no-office-apps' end office_tag,  */
        
        CASE 
            WHEN apps_installed LIKE '%uber%' OR apps_installed LIKE '%ola%' OR apps_installed LIKE '%blusmart%' OR apps_installed LIKE '%nammayatri%' OR apps_installed LIKE '%in drive%' 
            OR apps_installed LIKE '%quick ride%' OR apps_installed LIKE '%jugnoo%' 
        THEN 'RHA' 
            else 'NRHA' end as RHA_Check

        /*CASE 
            when apps_installed LIKE '%zomato%' OR apps_installed LIKE '%swiggy%' 
        THEN 'Zwigato' 
            else 'Non-Zwigato' end as Zwigato_check*/

            
    from 
        datasets.customer_apps_installed_snapshot
    )
    
    
    select
        coalesce(RHA_Check, 'UNKNOWN') RHA_Check,
        sum(case when week_start_date = '20241216' then gross_orders end ) w01_gross,
        sum(case when week_start_date = '20241223' then gross_orders end ) w02_gross,
        sum(case when week_start_date = '20241230' then gross_orders end ) w03_gross,
        sum(case when week_start_date = '20250106' then gross_orders end ) w04_gross,
        sum(case when week_start_date = '20250113' then gross_orders end ) w05_gross,
        sum(case when week_start_date = '20250120' then gross_orders end ) w06_gross,
        sum(case when week_start_date = '20250127' then gross_orders end ) w07_gross,
        sum(case when week_start_date = '20250203' then gross_orders end ) w08_gross,
        sum(case when week_start_date = '20250210' then gross_orders end ) w09_gross,
        sum(case when week_start_date = '20250217' then gross_orders end ) w10_gross,
        sum(case when week_start_date = '20250224' then gross_orders end ) w11_gross,
        sum(case when week_start_date = '20250303' then gross_orders end ) w12_gross,
        sum(case when week_start_date = '20250310' then gross_orders end ) w13_gross,
        
        sum(orders.gross_orders) as overall_gross,
        100.00*sum(gross_orders)/sum(sum(gross_orders))over() as pct,
        
        sum(case when week_start_date = '20241216' then net_orders end ) w01_net,
        sum(case when week_start_date = '20241223' then net_orders end ) w02_net,
        sum(case when week_start_date = '20241230' then net_orders end ) w03_net,
        sum(case when week_start_date = '20250106' then net_orders end ) w04_net,
        sum(case when week_start_date = '20250113' then net_orders end ) w05_net,
        sum(case when week_start_date = '20250120' then net_orders end ) w06_net,
        sum(case when week_start_date = '20250127' then net_orders end ) w07_net,
        sum(case when week_start_date = '20250203' then net_orders end ) w08_net,
        sum(case when week_start_date = '20250210' then net_orders end ) w09_net,
        sum(case when week_start_date = '20250217' then net_orders end ) w10_net,
        sum(case when week_start_date = '20250224' then net_orders end ) w11_net,
        sum(case when week_start_date = '20250303' then net_orders end ) w12_net,
        sum(case when week_start_date = '20250310' then net_orders end ) w13_net,
        
        sum(orders.net_orders) as overall_gross,
        100.00*sum(net_orders)/sum(sum(net_orders))over() as pct
      
    from
        base
    left join 
        orders
        on base.customer_id = orders.customer_id
    left join
        appo
        on base.customer_id = appo.userid
    group by 1
    order by 1 
'''
cx_data6 = pd.read_sql(cx_data6, conn1)
cx_data6.head()

,RHA_Check,w01_gross,w02_gross,w03_gross,w04_gross,w05_gross,w06_gross,w07_gross,w08_gross,w09_gross,w10_gross,w11_gross,w12_gross,w13_gross,overall_gross,pct,w01_net,w02_net,w03_net,w04_net,w05_net,w06_net,w07_net,w08_net,w09_net,w10_net,w11_net,w12_net,w13_net,overall_gross,pct
0,NRHA,838663,735692,810976,865074,862195,910934,905595,1018649,1068741,951439,971852,1038330,1063415,12041555,19.28,548623,522559,555232,573336,568018,574641,589878,631868,647282,630896,634523,654946,691053,7822855,19.92
1,RHA,2610293,2045877,2255446,2649202,2517934,2900844,2860262,3176659,3304003,2922739,2955192,3180142,3072417,36451010,58.36,1665048,1433183,1541684,1672499,1621501,1745114,1793116,1876233,1892782,1853671,1853702,1903355,1928712,22780600,58.02
2,UNKNOWN,986557,712240,791104,1023362,946066,1127062,1115294,1231686,1289920,1145471,1142357,1250411,1209737,13971267,22.37,628735,500941,548595,641114,613507,670570,697188,720080,728064,719239,711457,737983,740887,8658360,22.05


In [67]:
cx_data6

,RHA_Check,w01_gross,w02_gross,w03_gross,w04_gross,w05_gross,w06_gross,w07_gross,w08_gross,w09_gross,w10_gross,w11_gross,w12_gross,w13_gross,overall_gross,pct,w01_net,w02_net,w03_net,w04_net,w05_net,w06_net,w07_net,w08_net,w09_net,w10_net,w11_net,w12_net,w13_net,overall_gross,pct
0,NRHA,838663,735692,810976,865074,862195,910934,905595,1018649,1068741,951439,971852,1038330,1063415,12041555,19.28,548623,522559,555232,573336,568018,574641,589878,631868,647282,630896,634523,654946,691053,7822855,19.92
1,RHA,2610293,2045877,2255446,2649202,2517934,2900844,2860262,3176659,3304003,2922739,2955192,3180142,3072417,36451010,58.36,1665048,1433183,1541684,1672499,1621501,1745114,1793116,1876233,1892782,1853671,1853702,1903355,1928712,22780600,58.02
2,UNKNOWN,986557,712240,791104,1023362,946066,1127062,1115294,1231686,1289920,1145471,1142357,1250411,1209737,13971267,22.37,628735,500941,548595,641114,613507,670570,697188,720080,728064,719239,711457,737983,740887,8658360,22.05


### Zwigato_check

In [49]:
cx_data7 = f'''
with base as (
    select 
        customer_id,
        taxi_regularity_segment
    from 
        datasets.iallocator_customer_segments
    where 
        run_date = '2025-03-17'
        and taxi_lifetime_last_ride_city = 'Bangalore'
    ),
    
    orders as (
    
    select
        date_format(date_trunc('week', date_parse(yyyymmdd, '%Y%m%d')), '%Y%m%d') week_start_date,
        customer_id,
        count(distinct order_id) gross_orders,
        count(distinct case when order_status = 'dropped' then order_id end) net_orders
    from 
        orders.order_logs_fact
    where
        yyyymmdd >= '20241216'
        and yyyymmdd <= '20250316'
        and city_name = 'Bangalore'
        and service_obj_service_name in ('Auto','Auto Lite','Auto Pool','AutoPremium','CityAuto','Link','Bike Lite','Bike Pink','CabEconomy','Bike Metro', 'CabPremium')
    group by 1,2
    ),
    
    appo as (
    
    select
        userid,
            /* 
            case 
                when regexp_like(apps_installed, 'jira|miro|zoho|teams||intune|figma|slack|confluence|webex|github') = true 
                then 'office-apps' 
            else 'no-office-apps' end office_tag,  
        
        CASE 
            WHEN apps_installed LIKE '%uber%' OR apps_installed LIKE '%ola%' OR apps_installed LIKE '%blusmart%' OR apps_installed LIKE '%nammayatri%' OR apps_installed LIKE '%in drive%' 
            OR apps_installed LIKE '%quick ride%' OR apps_installed LIKE '%jugnoo%' 
        THEN 'RHA' 
            else 'NRHA' end as RHA_Check,
        */

        CASE 
            when apps_installed LIKE '%zomato%' OR apps_installed LIKE '%swiggy%' 
        THEN 'Zwigato' 
            else 'Non-Zwigato' end as Zwigato_check

            
    from 
        datasets.customer_apps_installed_snapshot
    )
    
    
    select
        coalesce(Zwigato_check, 'UNKNOWN') Zwigato_check,
        sum(case when week_start_date = '20241216' then gross_orders end ) w01_gross,
        sum(case when week_start_date = '20241223' then gross_orders end ) w02_gross,
        sum(case when week_start_date = '20241230' then gross_orders end ) w03_gross,
        sum(case when week_start_date = '20250106' then gross_orders end ) w04_gross,
        sum(case when week_start_date = '20250113' then gross_orders end ) w05_gross,
        sum(case when week_start_date = '20250120' then gross_orders end ) w06_gross,
        sum(case when week_start_date = '20250127' then gross_orders end ) w07_gross,
        sum(case when week_start_date = '20250203' then gross_orders end ) w08_gross,
        sum(case when week_start_date = '20250210' then gross_orders end ) w09_gross,
        sum(case when week_start_date = '20250217' then gross_orders end ) w10_gross,
        sum(case when week_start_date = '20250224' then gross_orders end ) w11_gross,
        sum(case when week_start_date = '20250303' then gross_orders end ) w12_gross,
        sum(case when week_start_date = '20250310' then gross_orders end ) w13_gross,
        
        sum(orders.gross_orders) as overall_gross,
        100.00*sum(gross_orders)/sum(sum(gross_orders))over() as pct,
        
        sum(case when week_start_date = '20241216' then net_orders end ) w01_net,
        sum(case when week_start_date = '20241223' then net_orders end ) w02_net,
        sum(case when week_start_date = '20241230' then net_orders end ) w03_net,
        sum(case when week_start_date = '20250106' then net_orders end ) w04_net,
        sum(case when week_start_date = '20250113' then net_orders end ) w05_net,
        sum(case when week_start_date = '20250120' then net_orders end ) w06_net,
        sum(case when week_start_date = '20250127' then net_orders end ) w07_net,
        sum(case when week_start_date = '20250203' then net_orders end ) w08_net,
        sum(case when week_start_date = '20250210' then net_orders end ) w09_net,
        sum(case when week_start_date = '20250217' then net_orders end ) w10_net,
        sum(case when week_start_date = '20250224' then net_orders end ) w11_net,
        sum(case when week_start_date = '20250303' then net_orders end ) w12_net,
        sum(case when week_start_date = '20250310' then net_orders end ) w13_net,
        
        sum(orders.net_orders) as overall_gross,
        100.00*sum(net_orders)/sum(sum(net_orders))over() as pct

    from
        base
    left join 
        orders
        on base.customer_id = orders.customer_id
    left join
        appo
        on base.customer_id = appo.userid
    group by 1
'''
cx_data7 = pd.read_sql(cx_data7, conn3)
cx_data7.head()

,Zwigato_check,w01_gross,w02_gross,w03_gross,w04_gross,w05_gross,w06_gross,w07_gross,w08_gross,w09_gross,w10_gross,w11_gross,w12_gross,w13_gross,overall_gross,pct,w01_net,w02_net,w03_net,w04_net,w05_net,w06_net,w07_net,w08_net,w09_net,w10_net,w11_net,w12_net,w13_net,overall_gross,pct
0,UNKNOWN,986557,712240,791104,1023362,946066,1127062,1115294,1231686,1289920,1145471,1142357,1250411,1209737,13971267,22.37,628735,500941,548595,641114,613507,670570,697188,720080,728064,719239,711457,737983,740887,8658360,22.05
1,Non-Zwigato,1028514,899674,973575,1045453,1043044,1116064,1109145,1236201,1292752,1153528,1164887,1240553,1256882,14560272,23.31,661155,626085,654864,679390,674628,693167,710457,755756,770350,751345,749247,768816,806347,9301607,23.69
2,Zwigato,2420442,1881895,2092847,2468823,2337085,2695714,2656712,2959107,3079992,2720650,2762157,2977919,2878950,33932293,54.32,1552516,1329657,1442052,1566445,1514891,1626588,1672537,1752345,1769714,1733222,1738978,1789485,1813418,21301848,54.26


In [69]:
cx_data7

,Zwigato_check,w01_gross,w02_gross,w03_gross,w04_gross,w05_gross,w06_gross,w07_gross,w08_gross,w09_gross,w10_gross,w11_gross,w12_gross,w13_gross,overall_gross,pct,w01_net,w02_net,w03_net,w04_net,w05_net,w06_net,w07_net,w08_net,w09_net,w10_net,w11_net,w12_net,w13_net,overall_gross,pct
0,UNKNOWN,986557,712240,791104,1023362,946066,1127062,1115294,1231686,1289920,1145471,1142357,1250411,1209737,13971267,22.37,628735,500941,548595,641114,613507,670570,697188,720080,728064,719239,711457,737983,740887,8658360,22.05
1,Non-Zwigato,1028514,899674,973575,1045453,1043044,1116064,1109145,1236201,1292752,1153528,1164887,1240553,1256882,14560272,23.31,661155,626085,654864,679390,674628,693167,710457,755756,770350,751345,749247,768816,806347,9301607,23.69
2,Zwigato,2420442,1881895,2092847,2468823,2337085,2695714,2656712,2959107,3079992,2720650,2762157,2977919,2878950,33932293,54.32,1552516,1329657,1442052,1566445,1514891,1626588,1672537,1752345,1769714,1733222,1738978,1789485,1813418,21301848,54.26


### Office Usecases

In [51]:
cx_data8 = f'''
with base as (
    select 
        customer_id,
        taxi_regularity_segment
    from 
        datasets.iallocator_customer_segments
    where 
        run_date = '2025-03-17'
        and taxi_lifetime_last_ride_city = 'Bangalore'
    ),
    
    orders as (
    
    select
        date_format(date_trunc('week', date_parse(yyyymmdd, '%Y%m%d')), '%Y%m%d') week_start_date,
        customer_id,
        count(distinct order_id) gross_orders,
        count(distinct case when order_status = 'dropped' then order_id end) net_orders
    from 
        orders.order_logs_fact
    where
        yyyymmdd >= '20241216'
        and yyyymmdd <= '20250316'
        and city_name = 'Bangalore'
        and service_obj_service_name in ('Auto','Auto Lite','Auto Pool','AutoPremium','CityAuto','Link','Bike Lite','Bike Pink','CabEconomy','Bike Metro', 'CabPremium')
    group by 1,2
    ),
    
    appo as (
    
    select
        userid,
            case 
                when regexp_like(apps_installed, 'jira|miro|zoho|teams|intune|figma|slack|confluence|webex|github') = true 
                then 'office-apps' 
            else 'no-office-apps' end office_tag    
        from 
        datasets.customer_apps_installed_snapshot
    )
    
    
    select
        coalesce(office_tag, 'UNKNOWN') office_tag,
        sum(case when week_start_date = '20241216' then gross_orders end ) w01_gross,
        sum(case when week_start_date = '20241223' then gross_orders end ) w02_gross,
        sum(case when week_start_date = '20241230' then gross_orders end ) w03_gross,
        sum(case when week_start_date = '20250106' then gross_orders end ) w04_gross,
        sum(case when week_start_date = '20250113' then gross_orders end ) w05_gross,
        sum(case when week_start_date = '20250120' then gross_orders end ) w06_gross,
        sum(case when week_start_date = '20250127' then gross_orders end ) w07_gross,
        sum(case when week_start_date = '20250203' then gross_orders end ) w08_gross,
        sum(case when week_start_date = '20250210' then gross_orders end ) w09_gross,
        sum(case when week_start_date = '20250217' then gross_orders end ) w10_gross,
        sum(case when week_start_date = '20250224' then gross_orders end ) w11_gross,
        sum(case when week_start_date = '20250303' then gross_orders end ) w12_gross,
        sum(case when week_start_date = '20250310' then gross_orders end ) w13_gross,
        
        sum(orders.gross_orders) as overall_gross,
        100.00*sum(gross_orders)/sum(sum(gross_orders))over() as pct,
        
        sum(case when week_start_date = '20241216' then net_orders end ) w01_net,
        sum(case when week_start_date = '20241223' then net_orders end ) w02_net,
        sum(case when week_start_date = '20241230' then net_orders end ) w03_net,
        sum(case when week_start_date = '20250106' then net_orders end ) w04_net,
        sum(case when week_start_date = '20250113' then net_orders end ) w05_net,
        sum(case when week_start_date = '20250120' then net_orders end ) w06_net,
        sum(case when week_start_date = '20250127' then net_orders end ) w07_net,
        sum(case when week_start_date = '20250203' then net_orders end ) w08_net,
        sum(case when week_start_date = '20250210' then net_orders end ) w09_net,
        sum(case when week_start_date = '20250217' then net_orders end ) w10_net,
        sum(case when week_start_date = '20250224' then net_orders end ) w11_net,
        sum(case when week_start_date = '20250303' then net_orders end ) w12_net,
        sum(case when week_start_date = '20250310' then net_orders end ) w13_net,
        
        sum(orders.net_orders) as overall_gross,
        100.00*sum(net_orders)/sum(sum(net_orders))over() as pct
      
    from
        base
    left join 
        orders
        on base.customer_id = orders.customer_id
    left join
        appo
        on base.customer_id = appo.userid
    group by 1
    order by 1 
'''
cx_data8 = pd.read_sql(cx_data8, conn3)
cx_data8.head()

,office_tag,w01_gross,w02_gross,w03_gross,w04_gross,w05_gross,w06_gross,w07_gross,w08_gross,w09_gross,w10_gross,w11_gross,w12_gross,w13_gross,overall_gross,pct,w01_net,w02_net,w03_net,w04_net,w05_net,w06_net,w07_net,w08_net,w09_net,w10_net,w11_net,w12_net,w13_net,overall_gross,pct
0,UNKNOWN,986557,712240,791104,1023362,946066,1127062,1115294,1231686,1289920,1145471,1142357,1250411,1209737,13971267,22.37,628735,500941,548595,641114,613507,670570,697188,720080,728064,719239,711457,737983,740887,8658360,22.05
1,no-office-apps,2287472,1940659,2112301,2307732,2284454,2476260,2442353,2722565,2840480,2531394,2560968,2716079,2716735,31939452,51.13,1469808,1355749,1432430,1492242,1479678,1528872,1558964,1649768,1679274,1638774,1634308,1671615,1735623,20327105,51.77
2,office-apps,1161484,840910,954121,1206544,1095675,1335518,1323504,1472743,1532264,1342784,1366076,1502393,1419097,16553113,26.50,743863,599993,664486,753593,709841,790883,824030,858333,860790,845793,853917,886686,884142,10276350,26.17


In [71]:
cx_data8

,office_tag,w01_gross,w02_gross,w03_gross,w04_gross,w05_gross,w06_gross,w07_gross,w08_gross,w09_gross,w10_gross,w11_gross,w12_gross,w13_gross,overall_gross,pct,w01_net,w02_net,w03_net,w04_net,w05_net,w06_net,w07_net,w08_net,w09_net,w10_net,w11_net,w12_net,w13_net,overall_gross,pct
0,UNKNOWN,986557,712240,791104,1023362,946066,1127062,1115294,1231686,1289920,1145471,1142357,1250411,1209737,13971267,22.37,628735,500941,548595,641114,613507,670570,697188,720080,728064,719239,711457,737983,740887,8658360,22.05
1,no-office-apps,2287472,1940659,2112301,2307732,2284454,2476260,2442353,2722565,2840480,2531394,2560968,2716079,2716735,31939452,51.13,1469808,1355749,1432430,1492242,1479678,1528872,1558964,1649768,1679274,1638774,1634308,1671615,1735623,20327105,51.77
2,office-apps,1161484,840910,954121,1206544,1095675,1335518,1323504,1472743,1532264,1342784,1366076,1502393,1419097,16553113,26.50,743863,599993,664486,753593,709841,790883,824030,858333,860790,845793,853917,886686,884142,10276350,26.17


### Age Group

In [55]:
cx_data9 = f'''
with base as (
    select 
        customer_id,
        taxi_regularity_segment
    from 
        datasets.iallocator_customer_segments
    where 
        run_date = '2025-03-17'
        and taxi_lifetime_last_ride_city = 'Bangalore'
    ),
    
    orders as (
    
    select
        date_format(date_trunc('week', date_parse(yyyymmdd, '%Y%m%d')), '%Y%m%d') week_start_date,
        customer_id,
        count(distinct order_id) gross_orders,
        count(distinct case when order_status = 'dropped' then order_id end) net_orders
    from 
        orders.order_logs_fact
    where
        yyyymmdd >= '20241216'
        and yyyymmdd <= '20250316'
        and city_name = 'Bangalore'
        and service_obj_service_name in ('Auto','Auto Lite','Auto Pool','AutoPremium','CityAuto','Link','Bike Lite','Bike Pink','CabEconomy','Bike Metro', 'CabPremium')
    group by 1,2
    ),
    
    age as (
    
    select
        customer_id,
        age_group
    from 
        hive.experiments_internal.customer_predicated_age_immutable
    )
    
    
    select
        coalesce(age_group, 'UNKNOWN') age_group,
        sum(case when week_start_date = '20241216' then gross_orders end ) w01_gross,
        sum(case when week_start_date = '20241223' then gross_orders end ) w02_gross,
        sum(case when week_start_date = '20241230' then gross_orders end ) w03_gross,
        sum(case when week_start_date = '20250106' then gross_orders end ) w04_gross,
        sum(case when week_start_date = '20250113' then gross_orders end ) w05_gross,
        sum(case when week_start_date = '20250120' then gross_orders end ) w06_gross,
        sum(case when week_start_date = '20250127' then gross_orders end ) w07_gross,
        sum(case when week_start_date = '20250203' then gross_orders end ) w08_gross,
        sum(case when week_start_date = '20250210' then gross_orders end ) w09_gross,
        sum(case when week_start_date = '20250217' then gross_orders end ) w10_gross,
        sum(case when week_start_date = '20250224' then gross_orders end ) w11_gross,
        sum(case when week_start_date = '20250303' then gross_orders end ) w12_gross,
        sum(case when week_start_date = '20250310' then gross_orders end ) w13_gross,
        
        sum(orders.gross_orders) as overall_gross,
        100.00*sum(gross_orders)/sum(sum(gross_orders))over() as pct,
        
        sum(case when week_start_date = '20241216' then net_orders end ) w01_net,
        sum(case when week_start_date = '20241223' then net_orders end ) w02_net,
        sum(case when week_start_date = '20241230' then net_orders end ) w03_net,
        sum(case when week_start_date = '20250106' then net_orders end ) w04_net,
        sum(case when week_start_date = '20250113' then net_orders end ) w05_net,
        sum(case when week_start_date = '20250120' then net_orders end ) w06_net,
        sum(case when week_start_date = '20250127' then net_orders end ) w07_net,
        sum(case when week_start_date = '20250203' then net_orders end ) w08_net,
        sum(case when week_start_date = '20250210' then net_orders end ) w09_net,
        sum(case when week_start_date = '20250217' then net_orders end ) w10_net,
        sum(case when week_start_date = '20250224' then net_orders end ) w11_net,
        sum(case when week_start_date = '20250303' then net_orders end ) w12_net,
        sum(case when week_start_date = '20250310' then net_orders end ) w13_net,
        
        sum(orders.net_orders) as overall_gross,
        100.00*sum(net_orders)/sum(sum(net_orders))over() as pct

      
    from
        base
    left join 
        orders
        on base.customer_id = orders.customer_id
    left join
        age
        on base.customer_id = age.customer_id
    group by 1
    order by 1
'''
cx_data9 = pd.read_sql(cx_data9, conn3)
cx_data9.head(10)

,age_group,w01_gross,w02_gross,w03_gross,w04_gross,w05_gross,w06_gross,w07_gross,w08_gross,w09_gross,w10_gross,w11_gross,w12_gross,w13_gross,overall_gross,pct,w01_net,w02_net,w03_net,w04_net,w05_net,w06_net,w07_net,w08_net,w09_net,w10_net,w11_net,w12_net,w13_net,overall_gross,pct
0,21-25,832826,680389,769805,850094,844398,913254,906495,1010276,1025499,881975,904883,949630,948000,11517524,18.44,509873,456853,499531,522643,521816,534225,548593,577548,568701,543323,547984,552993,573680,6957763,17.72
1,26-30,766401,620098,697637,797414,747518,851945,843607,943972,960856,820018,844375,914220,871135,10679196,17.10,478597,428342,465846,491231,472351,499278,517382,543320,535370,513466,520225,529711,535016,6530135,16.63
2,31-35,625752,505393,545783,641017,606643,697456,685168,753744,766298,669529,665593,718425,687794,8568595,13.72,406287,359524,379016,411077,397284,425582,436771,452920,444428,428970,424217,434698,437711,5438485,13.85
3,36-45,891850,712011,764579,891438,856682,980315,967534,1054490,1060169,949727,930541,984010,950496,11993842,19.20,601464,522688,550682,601850,583558,630934,645042,666384,649062,631614,615268,628920,629256,7956722,20.27
4,Above-45,191781,150988,163405,191760,186640,213620,210878,232390,232229,208631,204708,215333,210842,2613205,4.18,129640,111252,118444,129918,127440,137886,141381,147803,142852,139833,135707,138500,140540,1741196,4.43
5,Below-21,64585,53590,59780,65001,64453,69105,68106,75516,73506,64158,65245,68000,68190,859235,1.38,39670,35839,38924,40314,39501,40691,41541,43975,41438,39661,39714,40278,41576,523122,1.33
6,UNKNOWN,1062318,771340,856537,1100914,1019861,1213145,1199363,1356606,1544107,1425611,1454056,1619265,1609112,16232235,25.99,676875,542185,593068,689916,661076,721729,749472,796231,886277,906939,916567,971184,1002873,10114392,25.76


In [73]:
cx_data9

,age_group,w01_gross,w02_gross,w03_gross,w04_gross,w05_gross,w06_gross,w07_gross,w08_gross,w09_gross,w10_gross,w11_gross,w12_gross,w13_gross,overall_gross,pct,w01_net,w02_net,w03_net,w04_net,w05_net,w06_net,w07_net,w08_net,w09_net,w10_net,w11_net,w12_net,w13_net,overall_gross,pct
0,21-25,832826,680389,769805,850094,844398,913254,906495,1010276,1025499,881975,904883,949630,948000,11517524,18.44,509873,456853,499531,522643,521816,534225,548593,577548,568701,543323,547984,552993,573680,6957763,17.72
1,26-30,766401,620098,697637,797414,747518,851945,843607,943972,960856,820018,844375,914220,871135,10679196,17.10,478597,428342,465846,491231,472351,499278,517382,543320,535370,513466,520225,529711,535016,6530135,16.63
2,31-35,625752,505393,545783,641017,606643,697456,685168,753744,766298,669529,665593,718425,687794,8568595,13.72,406287,359524,379016,411077,397284,425582,436771,452920,444428,428970,424217,434698,437711,5438485,13.85
3,36-45,891850,712011,764579,891438,856682,980315,967534,1054490,1060169,949727,930541,984010,950496,11993842,19.20,601464,522688,550682,601850,583558,630934,645042,666384,649062,631614,615268,628920,629256,7956722,20.27
4,Above-45,191781,150988,163405,191760,186640,213620,210878,232390,232229,208631,204708,215333,210842,2613205,4.18,129640,111252,118444,129918,127440,137886,141381,147803,142852,139833,135707,138500,140540,1741196,4.43
5,Below-21,64585,53590,59780,65001,64453,69105,68106,75516,73506,64158,65245,68000,68190,859235,1.38,39670,35839,38924,40314,39501,40691,41541,43975,41438,39661,39714,40278,41576,523122,1.33
6,UNKNOWN,1062318,771340,856537,1100914,1019861,1213145,1199363,1356606,1544107,1425611,1454056,1619265,1609112,16232235,25.99,676875,542185,593068,689916,661076,721729,749472,796231,886277,906939,916567,971184,1002873,10114392,25.76


### Office Use-Cases

In [61]:
cx_data11 = f'''
with base as (
    select 
        customer_id,
        taxi_regularity_segment
    from 
        datasets.iallocator_customer_segments
    where 
        run_date = '2025-03-17'
        and taxi_lifetime_last_ride_city = 'Bangalore'
    ),
    
    use_case as (
    
    select
        hex_12,
        primary_tag
    from 
        experiments_internal.combined_geo_usecase_hex_12_level
    where 
        current_city = 'Bangalore'
        and primary_tag = 'office'
    ),
    
    orders as (
    
    select
        date_format(date_trunc('week', date_parse(yyyymmdd, '%Y%m%d')), '%Y%m%d') week_start_date,
        customer_id,
        case 
        when pickup_location_hex_12 in (select distinct hex_12 from use_case) then 'office'
        when drop_location_hex_12 in (select distinct hex_12 from use_case) then 'office'
        else 'non-office' end office_use_case,
        count(distinct order_id) gross_orders,
        count(distinct case when order_status = 'dropped' then order_id end) net_orders
    from 
        orders.order_logs_snapshot
    where
        yyyymmdd >= '20241216'
        and yyyymmdd <= '20250316'
        and city_name = 'Bangalore'
        and service_obj_service_name in ('Auto','Auto Lite','Auto Pool','AutoPremium','CityAuto','Link','Bike Lite','Bike Pink','CabEconomy','Bike Metro', 'CabPremium')
    group by 1,2,3
    )
    
    select
        coalesce(office_use_case, 'non-office') office_use_case,
        sum(case when week_start_date = '20241216' then gross_orders end ) w01_gross,
        sum(case when week_start_date = '20241223' then gross_orders end ) w02_gross,
        sum(case when week_start_date = '20241230' then gross_orders end ) w03_gross,
        sum(case when week_start_date = '20250106' then gross_orders end ) w04_gross,
        sum(case when week_start_date = '20250113' then gross_orders end ) w05_gross,
        sum(case when week_start_date = '20250120' then gross_orders end ) w06_gross,
        sum(case when week_start_date = '20250127' then gross_orders end ) w07_gross,
        sum(case when week_start_date = '20250203' then gross_orders end ) w08_gross,
        sum(case when week_start_date = '20250210' then gross_orders end ) w09_gross,
        sum(case when week_start_date = '20250217' then gross_orders end ) w10_gross,
        sum(case when week_start_date = '20250224' then gross_orders end ) w11_gross,
        sum(case when week_start_date = '20250303' then gross_orders end ) w12_gross,
        sum(case when week_start_date = '20250310' then gross_orders end ) w13_gross,
        
        sum(orders.gross_orders) as overall_gross,
        100.00*sum(gross_orders)/sum(sum(gross_orders))over() as pct,
        
        sum(case when week_start_date = '20241216' then net_orders end ) w01_net,
        sum(case when week_start_date = '20241223' then net_orders end ) w02_net,
        sum(case when week_start_date = '20241230' then net_orders end ) w03_net,
        sum(case when week_start_date = '20250106' then net_orders end ) w04_net,
        sum(case when week_start_date = '20250113' then net_orders end ) w05_net,
        sum(case when week_start_date = '20250120' then net_orders end ) w06_net,
        sum(case when week_start_date = '20250127' then net_orders end ) w07_net,
        sum(case when week_start_date = '20250203' then net_orders end ) w08_net,
        sum(case when week_start_date = '20250210' then net_orders end ) w09_net,
        sum(case when week_start_date = '20250217' then net_orders end ) w10_net,
        sum(case when week_start_date = '20250224' then net_orders end ) w11_net,
        sum(case when week_start_date = '20250303' then net_orders end ) w12_net,
        sum(case when week_start_date = '20250310' then net_orders end ) w13_net,
        
        sum(orders.net_orders) as overall_gross,
        100.00*sum(net_orders)/sum(sum(net_orders))over() as pct
        
    from
        base
    left join 
        orders
        on base.customer_id = orders.customer_id
    group by 1
    order by 1
'''
cx_data11 = pd.read_sql(cx_data11, conn1)
cx_data11.head(10)

,office_use_case,w01_gross,w02_gross,w03_gross,w04_gross,w05_gross,w06_gross,w07_gross,w08_gross,w09_gross,w10_gross,w11_gross,w12_gross,w13_gross,overall_gross,pct,w01_net,w02_net,w03_net,w04_net,w05_net,w06_net,w07_net,w08_net,w09_net,w10_net,w11_net,w12_net,w13_net,overall_gross,pct
0,non-office,3757198,3084644,3415534,3785345,3698478,4100249,4103338,4536983,4700796,4176593,4254353,4509548,4467628,52590687,84.19,2447354,2178490,2351068,2478663,2434847,2563861,2644707,2774820,2806257,2741052,2749270,2814135,2886677,33871201,86.27
1,office,678315,409165,441992,752293,627717,838591,777813,890011,961868,843056,815048,959335,877941,9873145,15.81,395052,278193,294443,408286,368179,426464,435475,453361,461871,462754,450412,482149,473975,5390614,13.73


In [75]:
cx_data11

,office_use_case,w01_gross,w02_gross,w03_gross,w04_gross,w05_gross,w06_gross,w07_gross,w08_gross,w09_gross,w10_gross,w11_gross,w12_gross,w13_gross,overall_gross,pct,w01_net,w02_net,w03_net,w04_net,w05_net,w06_net,w07_net,w08_net,w09_net,w10_net,w11_net,w12_net,w13_net,overall_gross,pct
0,non-office,3757198,3084644,3415534,3785345,3698478,4100249,4103338,4536983,4700796,4176593,4254353,4509548,4467628,52590687,84.19,2447354,2178490,2351068,2478663,2434847,2563861,2644707,2774820,2806257,2741052,2749270,2814135,2886677,33871201,86.27
1,office,678315,409165,441992,752293,627717,838591,777813,890011,961868,843056,815048,959335,877941,9873145,15.81,395052,278193,294443,408286,368179,426464,435475,453361,461871,462754,450412,482149,473975,5390614,13.73
